# 1 · Los datos: de las fuentes oficiales al almacén
**Torre del Caribe** · Grandes volúmenes de datos (criterio 2) · Autor: **Brandon Uriel García Sánchez** · Equipo: Maribel
Mondragón Mercado, Jesús Ramírez Isidro, Enrique González Ortega · UNRC, LCDN 5° semestre 2026-2

Este notebook recorre el camino de los datos: **qué se descargó** (caja de bronce), **cómo se limpió con Spark** (caja de
plata), **cómo se guarda y se consulta** (DuckDB y el modelo estrella) y **qué no existe** (huecos declarados).

| | |
|---|---|
| **Cómo correrlo** | Python 3.11 con `pip install -r requirements.txt` (en la carpeta de la entrega). Para Spark (notebooks 1 y 3), además Java 17: `winget install Microsoft.OpenJDK.17`. Abrir en Jupyter o VS Code y elegir *Ejecutar todo*. |
| **Datos** | Los lee de `datos/` (tablas limpias y resultados). Ya viene ejecutado: se puede leer sin correrlo. |
| **Código** | Va dentro del notebook: cada celda que empieza con `%%modulo` es un archivo del proyecto, completo. |

| **El crudo** | Los 353 archivos crudos (814 MB) no van en la entrega; va su manifiesto con huella SHA-256 y dirección de origen. Si la carpeta `datos/bronze` está completa, las celdas de limpieza vuelven a correr con Spark; si no, se saltan y se usan las tablas limpias que ya vienen. |

In [1]:
%matplotlib inline
# Preparación: se corre una vez, al inicio. Encuentra la carpeta de la entrega y define %%modulo, la instrucción que
# convierte una celda de código en un módulo del paquete `torre` (así el notebook no necesita archivos .py).
import os, sys, types, warnings
from pathlib import Path
from IPython.core.magic import register_cell_magic
warnings.filterwarnings("ignore")

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos" / "silver").is_dir())
FIGURAS = RAIZ / "2_Notebooks" / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)


def _paquete(nombre):
    if nombre not in sys.modules:
        m = types.ModuleType(nombre); m.__path__ = []
        sys.modules[nombre] = m
        if "." in nombre:
            padre, hijo = nombre.rsplit(".", 1)
            setattr(_paquete(padre), hijo, m)
    return sys.modules[nombre]


@register_cell_magic
def modulo(linea, celda):
    """%%modulo torre.radar.indice → ejecuta la celda como el módulo torre.radar.indice."""
    nombre = linea.strip()
    padre, hijo = nombre.rsplit(".", 1)
    m = types.ModuleType(nombre)
    # Ruta simbólica: el código calcula la raíz del proyecto con Path(__file__).parents[3], y aquí da la carpeta de entrega.
    m.__file__ = str(RAIZ / "codigo" / Path(*nombre.split(".")).with_suffix(".py"))
    sys.modules[nombre] = m
    setattr(_paquete(padre), hijo, m)
    try:  # para que Spark pueda mandar a sus trabajadores las funciones definidas aquí
        from pyspark import cloudpickle
        cloudpickle.register_pickle_by_value(m)
    except Exception:
        pass
    exec(compile(celda, m.__file__, "exec"), m.__dict__)


import pandas as pd
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
print("Carpeta de la entrega:", RAIZ)

Carpeta de la entrega: C:\Users\Uriel\Desktop\PP 5to semestre CASCO SANTO TOMAS\Entregable


## 1. Qué se descargó (la caja de bronce)
Cada archivo se descargó con un programa, sin modificarlo, y quedó anotado en el **manifiesto**: su huella SHA-256 (si se
cambia una coma, la huella cambia), su tamaño, su dirección de origen, la fecha y cuántos registros tiene.

In [2]:
m = pd.read_csv(RAIZ / "datos" / "bronze" / "MANIFIESTO.csv")
# La evidencia del sargazo y las capturas de los costos de anuncios también están en el manifiesto, pero no son fuentes.
oficiales = m[~m.fuente.isin(["Evidencia sargazo", "D13 Benchmarks"])]
print(f"Fuentes oficiales: {len(oficiales)} archivos · {oficiales.filas.sum():,.0f} registros · {oficiales.bytes.sum() / 1e6:,.1f} MB")
(oficiales.groupby("fuente").agg(archivos=("archivo", "size"), registros=("filas", "sum"), MB=("bytes", lambda b: round(b.sum() / 1e6, 1)))
 .sort_values("registros", ascending=False))

Fuentes oficiales: 353 archivos · 8,134,802 registros · 824.1 MB


,archivos,registros,MB
fuente,,,
D6 DENUE INEGI,33,6138075.0,574.2
D8 Open-Meteo,128,767016.0,24.5
D3 DataTur BD_Nacionalidad,1,521364.0,19.0
D4 DataTur Compendio 2024,1,295191.0,31.3
D5 Rest-Mex 2025,1,208051.0,89.0
D4 DataTur BdINAH,1,71562.0,3.1
D9 HURDAT2 NOAA,1,57512.0,7.1
D2 DataTur ocupación hotelera,166,34164.0,67.8
D4 DataTur DB_AFAC,1,19578.0,0.7


## 2. Cómo se descargó
Estas funciones bajan los datos de las fuentes oficiales. **No se vuelven a correr aquí** (descargan cientos de MB de
internet); se muestran para que se vea exactamente cómo se obtuvo cada archivo y cómo se registró en el manifiesto.

### El código de la descarga
Cada celda de código de abajo es un archivo del paquete `torre`, completo y sin cambios (con su encabezado: qué hace, por qué así, de qué datos sale y a qué decisión alimenta). Al correrla, la celda queda registrada como ese módulo y las celdas siguientes lo usan.

- **`torre.base.manifiesto`** — Registra cada archivo crudo descargado (Bronze) en datos/bronze/MANIFIESTO.csv con su huella SHA-256, tamaño, fuente, URL, fecha de descarga y número de filas.
- **`torre.base.ingesta_siturq`** — Descarga de SITUR-Q (sistema estatal de información turística de Quintana Roo) los indicadores elegidos, para cada destino, zona y año 2019–2026, y los guarda crudos en datos/bronze/siturq/<fecha>/ con su registro en el manifiesto.
- **`torre.base.ingesta_datatur`** — Descarga de DataTur (SECTUR) los archivos oficiales: ocupación hotelera semanal y mensual, llegadas por nacionalidad y aeropuerto, visitantes del INAH, tráfico aéreo (AFAC), cruceros y el Compendio Estadístico 2024. Los guarda crudos en datos/bronze/datatur/<fecha>/ y los registra en el manifiesto con su huella SHA-256 y su número de filas.
- **`torre.base.ingesta_abiertas`** — Descarga las fuentes abiertas restantes del inventario y las registra en el manifiesto: D5 Rest-Mex 2025, D6 DENUE (32 estados), D7 Censo 2020 ITER Q. Roo, D8 Open-Meteo (clima horario y diario de 8 puntos), D9 HURDAT2, D10 FRED, D11 ENDUTIH 2025 (PDF) y D12 GeoJSON.
- **`torre.base.ingesta_benchmarks`** — Extrae con un navegador automatizado (Playwright + Chromium) las tablas de costos y desempeño publicitario por industria de WordStream 2025 y LocaliQ 2026 (Google Ads/búsqueda y Facebook). Guarda el HTML completo, una captura de pantalla de página completa y todas las tablas en un CSV largo, y registra todo en el manifiesto como fuente D13.

In [3]:
%%modulo torre.base.manifiesto
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Registra cada archivo crudo descargado (Bronze) en datos/bronze/MANIFIESTO.csv con su huella
#                    SHA-256, tamaño, fuente, URL, fecha de descarga y número de filas.
# Por qué así:       Regla de oro 2 (toda cifra es rastreable) y el incidente de Big Data (qué se guarda y por qué).
#                    Con la huella cualquiera puede comprobar que el archivo es el original de la fuente oficial.
#                    Alternativa descartada: guardar solo los archivos sin registro. Así no hay forma de probar
#                    de dónde vino un número ni si alguien cambió el archivo después.
# Datos de entrada:  Los archivos que escriben las funciones de ingesta.
# Alimenta a:        La trazabilidad de todas las cifras de la campaña y del informe.

import csv
import hashlib
from datetime import datetime, timezone
from pathlib import Path

RAIZ = Path(__file__).resolve().parents[3]
BRONZE = RAIZ / "datos" / "bronze"
MANIFIESTO = BRONZE / "MANIFIESTO.csv"
COLUMNAS = ["fuente", "archivo", "sha256", "bytes", "filas", "url", "descargado_utc", "nota"]


def sha256_de(ruta: Path) -> str:
    """Huella SHA-256 del archivo, leída en bloques de 1 MB para no cargar archivos grandes en memoria."""
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def registrar(fuente: str, ruta: Path, url: str, filas: int | None = None, nota: str = "") -> dict:
    """Agrega (o actualiza) la fila de un archivo en el manifiesto y la devuelve.

    - fuente: código del inventario (p. ej. "D1 SITUR-Q").
    - filas:  número de registros que trae el archivo (None si no aplica, p. ej. un PDF).
    Si el archivo ya estaba registrado, se reemplaza su fila; así el manifiesto no duplica entradas.
    """
    ruta = Path(ruta)
    fila = {
        "fuente": fuente,
        "archivo": ruta.relative_to(RAIZ).as_posix(),
        "sha256": sha256_de(ruta),
        "bytes": ruta.stat().st_size,
        "filas": "" if filas is None else filas,
        "url": url,
        "descargado_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "nota": nota,
    }
    existentes = []
    if MANIFIESTO.exists():
        with open(MANIFIESTO, newline="", encoding="utf-8") as f:
            existentes = [r for r in csv.DictReader(f) if r["archivo"] != fila["archivo"]]
    MANIFIESTO.parent.mkdir(parents=True, exist_ok=True)
    with open(MANIFIESTO, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUMNAS)
        w.writeheader()
        w.writerows(existentes + [fila])
    return fila


In [4]:
%%modulo torre.base.ingesta_siturq
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Descarga de SITUR-Q (sistema estatal de información turística de Quintana Roo) los
#                    indicadores elegidos, para cada destino, zona y año 2019–2026, y los guarda crudos en
#                    datos/bronze/siturq/<fecha>/ con su registro en el manifiesto.
# Por qué así:       SITUR-Q es la única fuente oficial con datos del SUR (Chetumal, Bacalar, Mahahual, Maya Ka'an).
#                    Su tablero público consulta una API con un token que la propia página entrega a cualquier
#                    visitante. Aquí se reproduce exactamente esa consulta del navegador (verificado el
#                    27-sep-2026: Bacalar, 145 hoteles en 2025).
#                    Se guarda la respuesta COMPLETA de la API, sin tocarla: limpiar es trabajo de Silver.
#                    Alternativa descartada: exportar a mano con el botón "Exportar Excel", indicador por
#                    indicador. No es reproducible y serían cientos de clics.
# Datos de entrada:  D1 SITUR-Q (https://siturq.gob.mx/indicadores-turisticos y su API returq.siturq.gob.mx).
# Alimenta a:        A1 Radar (presión y capacidad del sur), A3 Pronóstico (series mensuales) y la selección de
#                    regiones (docs/decisiones/01-regiones.md).

import html
import json
import re
import time
from datetime import date
from pathlib import Path

import requests

from torre.base.manifiesto import BRONZE, registrar

PAGINA = "https://siturq.gob.mx/indicadores-turisticos"
API = "https://returq.siturq.gob.mx/api/charts/getCharData"
# La API rechaza peticiones que no vienen "desde" su página ("Origin not allowed"), así que se envían los mismos
# encabezados que manda el navegador.
ENCABEZADOS = {"Origin": "https://siturq.gob.mx", "Referer": PAGINA, "User-Agent": "Mozilla/5.0 (proyecto escolar UNRC)"}

# Indicadores elegidos por Brandon el 28-sep-2026 (propuesta de la Fase 1). id de SITUR-Q → nombre corto.
INDICADORES = {
    "5ff84749-c44b-48b3-934e-b11c1114afe1": "ocupacion_hotelera",
    "5a60d7fe-ec6f-45e9-87ac-f0951926ddc3": "habitaciones",
    "0cb48138-93f7-442e-a573-2b1d9d85e1a2": "centros_hospedaje",
    "c2dbcbc1-291e-4e5b-98f1-9dde4ec8b51b": "cruceristas",
    "74e66b9b-d5ec-4361-bf0f-0344b5c2a45a": "frontera_belice",
    "a0859e5e-77d8-4fd2-92b2-4d02eda16534": "tren_maya_movimiento",
    "a081b770-672c-4353-926d-23f992f65ebb": "tren_maya_descenso",
    "321012c6-7730-4651-870a-87e772ecd290": "aereos_llegadas",
    "9c95b3c9-d11b-40ea-bc8c-05f7c19cdb73": "zonas_arqueologicas",
    "1b01eaf7-9e50-4b34-a911-665fc90fe6fd": "afluencia_turistas",
    "a002e47e-1db2-4e37-b09a-0191018c2556": "turista_afluencia",
    "ce1590d7-7ea5-44db-80a8-4ecb77ee0a94": "derrama_visitantes",
    "5942ad3c-ad95-45e7-8f34-bb074510d7cd": "derrama_turistas",
}


def leer_catalogo() -> dict:
    """Lee la página pública y extrae el token, los destinos y las zonas tal como los presenta SITUR-Q.

    Devuelve {"token": ..., "unidades": [...], "html": ...}. Cada unidad trae el cuerpo exacto que espera la API
    en el campo destinationsByZone:
      - destino suelto (Cancún, Cozumel...):         isGroup = False
      - miembro de una zona (Chetumal, Tulum...):     se consulta como destino suelto con su propio id
      - zona (Riviera Maya, Grand Costa Maya):        isGroup = True, con la lista de sus miembros
      - zona especial (Caribe Mexicano = todo el estado): isSpecial = True
    """
    pagina = requests.get(PAGINA, headers=ENCABEZADOS, timeout=60).text
    token = re.search(r'const token = "([^"]+)"', pagina).group(1)

    entradas = []
    for m in re.finditer(r"<input[^>]*>", pagina):
        t = m.group(0)
        attr = lambda nombre: (re.search(r"\s" + nombre + r'="([^"]*)"', t) or [None, ""])[1]
        if "data-special-zone" in t:
            tipo = "zona_especial"
        elif "data-zone-length" in t:
            tipo = "zona"
        elif "data-zone-destination" in t:
            tipo = "miembro"
        elif "data-destination" in t:
            tipo = "destino"
        else:
            continue
        # \sid= evita confundir el id propio con data-zone-id (error que se detectó al explorar la página)
        entradas.append({"tipo": tipo, "id": attr("id"), "nombre": html.unescape(attr("name")), "zona": attr("data-zone-id")})

    unidades = []
    for e in entradas:
        if e["tipo"] in ("destino", "miembro"):
            cuerpo = [{"groupId": e["id"], "name": e["nombre"], "isGroup": False, "isSpecial": False, "isSelected": True}]
        elif e["tipo"] == "zona":
            miembros = [{"destinationId": m["id"], "name": m["nombre"], "isSelected": True}
                        for m in entradas if m["tipo"] == "miembro" and m["zona"] == e["id"]]
            cuerpo = [{"groupId": e["id"], "name": e["nombre"], "isGroup": True, "isSpecial": False,
                       "isSelected": True, "lstDestinations": miembros}]
        else:
            cuerpo = [{"groupId": e["id"], "name": e["nombre"], "isGroup": True, "isSpecial": True,
                       "isSelected": True, "lstDestinations": []}]
        unidades.append({**e, "cuerpo": cuerpo})
    return {"token": token, "unidades": unidades, "html": pagina}


def consultar(token: str, id_indicador: str, unidad: dict, anio: int, intentos: int = 3) -> dict:
    """Pide a la API un indicador para una unidad (destino o zona) y un año completo (meses 1 a 12).

    Reintenta hasta 3 veces si la red falla. Devuelve la respuesta JSON tal cual.
    """
    formulario = {
        "indicators": json.dumps([id_indicador]),
        "isOnline": "false",
        "isAccumulated": "false",
        "typeoptiongraphic": "",
        "isDestinationComparison": "false",
        "destinationsByZone": json.dumps(unidad["cuerpo"], ensure_ascii=False),
        "conditions": json.dumps([[["month", ">=", 1], ["year", "=", anio], ["month", "<=", 12], ["year", "=", anio]]]),
    }
    for intento in range(intentos):
        try:
            r = requests.post(API, headers={**ENCABEZADOS, "Authorization": f"Bearer {token}"},
                              files={k: (None, v) for k, v in formulario.items()}, timeout=60)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            if intento == intentos - 1:
                raise
            time.sleep(2 * (intento + 1))


def descargar_siturq(anios=range(2019, 2027), pausa: float = 0.2) -> list[dict]:
    """Descarga todos los INDICADORES × unidades × años y guarda un archivo JSON crudo por indicador.

    Cada archivo contiene una lista de registros {indicador, unidad, tipo, anio, respuesta}, donde "respuesta" es
    exactamente lo que devolvió la API. La pausa entre peticiones evita saturar el servidor público.
    Devuelve las filas del manifiesto que se registraron.
    """
    catalogo = leer_catalogo()
    carpeta = BRONZE / "siturq" / date.today().isoformat()
    carpeta.mkdir(parents=True, exist_ok=True)

    # Evidencia: la página tal como estaba el día de la descarga (de ahí salen los ids).
    pagina = carpeta / "_pagina_indicadores.html"
    pagina.write_text(catalogo["html"], encoding="utf-8")
    registrar("D1 SITUR-Q", pagina, PAGINA, nota="página pública de la que salen el token y los ids")

    registros_manifiesto = []
    for id_ind, nombre in INDICADORES.items():
        if (carpeta / f"{nombre}.json").exists():  # reanudación: ese indicador ya se bajó hoy
            print(f"{nombre:22s} ya estaba, se omite")
            continue
        bloque, filas, fallidas = [], 0, 0
        for unidad in catalogo["unidades"]:
            for anio in anios:
                base = {"indicador": nombre, "id_indicador": id_ind, "unidad": unidad["nombre"],
                        "tipo_unidad": unidad["tipo"], "anio": anio}
                try:
                    resp = consultar(catalogo["token"], id_ind, unidad, anio)
                    filas += len(resp.get("data") or [])
                    bloque.append({**base, "respuesta": resp})
                except requests.RequestException as e:
                    # Regla de oro 5: el hueco se declara, no se rellena. Queda el error exacto del servidor.
                    fallidas += 1
                    bloque.append({**base, "respuesta": None, "error": repr(e)})
                time.sleep(pausa)
        archivo = carpeta / f"{nombre}.json"
        archivo.write_text(json.dumps(bloque, ensure_ascii=False), encoding="utf-8")
        nota = f"{len(catalogo['unidades'])} unidades × {len(list(anios))} años"
        if fallidas:
            nota += f"; {fallidas} consultas fallaron en el servidor (hueco declarado)"
        registros_manifiesto.append(registrar("D1 SITUR-Q", archivo, API, filas=filas, nota=nota))
        print(f"{nombre:22s} {filas:6d} filas, {fallidas} fallidas → {archivo.name}")
    return registros_manifiesto


In [5]:
%%modulo torre.base.ingesta_datatur
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Descarga de DataTur (SECTUR) los archivos oficiales: ocupación hotelera semanal y mensual,
#                    llegadas por nacionalidad y aeropuerto, visitantes del INAH, tráfico aéreo (AFAC), cruceros y
#                    el Compendio Estadístico 2024. Los guarda crudos en datos/bronze/datatur/<fecha>/ y los
#                    registra en el manifiesto con su huella SHA-256 y su número de filas.
# Por qué así:       La lista de archivos se lee de las propias páginas de DataTur (no se inventan nombres), así
#                    que si SECTUR publica una semana nueva se descarga sola.
#                    Verificado el 27-sep-2026: 135 zips semanales (2024-S01 → 2026-S31) y 31 mensuales
#                    (2024-01 → 2026-07). BD_Nacionalidad trae ~521,364 filas; BdINAH incluye las zonas
#                    arqueológicas de Q. Roo.
#                    Alternativa descartada: descargar a mano desde el navegador, que no es reproducible.
# Datos de entrada:  D2, D2m, D3 y D4 del inventario (https://datatur.sectur.gob.mx).
# Alimenta a:        A1 Radar (ocupación semanal del norte, presión INAH), buyer persona (nacionalidad y sexo por
#                    aeropuerto) y A3 Pronóstico (series mensuales).

import io
import re
import zipfile
from datetime import date

import openpyxl
import requests

from torre.base.manifiesto import BRONZE, registrar

BASE = "https://datatur.sectur.gob.mx"
UA = {"User-Agent": "Mozilla/5.0 (proyecto escolar UNRC)"}

# Páginas de DataTur donde viven los enlaces a los archivos, y qué fuente del inventario es cada una.
PAGINAS = {
    "hoteleria": ("/SitePages/hoteleria.aspx", "D2 DataTur ocupación hotelera"),
    "nacionalidad": ("/SitePages/upmnacionalidad.aspx", "D3 DataTur BD_Nacionalidad"),
    "inah": ("/SitePages/museosyzonasarqueologicas.aspx", "D4 DataTur BdINAH"),
    "afac": ("/SitePages/afac.aspx", "D4 DataTur DB_AFAC"),
    "cruceros": ("/SitePages/cruceros.aspx", "D4 DataTur cruceros"),
    "compendio": ("/SitePages/compendioestadistico.aspx", "D4 DataTur Compendio 2024"),
}


def enlaces_de(pagina: str) -> list[str]:
    """Devuelve las rutas de los .zip/.xlsx publicados en una página de DataTur, sin repetir y en orden."""
    texto = requests.get(BASE + pagina, headers=UA, timeout=60).text
    rutas = re.findall(r'href="(/Documentoscompartidos/[^"]+\.(?:zip|xlsx))"', texto, flags=re.I)
    return sorted(set(rutas))


def contar_filas(contenido: bytes, nombre: str) -> int | None:
    """Cuenta las filas de todas las hojas de los Excel dentro de un zip (o de un .xlsx suelto).

    Es el conteo "crudo": incluye encabezados y notas. El conteo limpio se hace en Silver.
    Se usa read_only para no cargar libros grandes completos en memoria.
    """
    def filas_xlsx(datos: bytes) -> int:
        libro = openpyxl.load_workbook(io.BytesIO(datos), read_only=True, data_only=True)
        total = sum((hoja.max_row or 0) for hoja in libro.worksheets)
        libro.close()
        return total

    try:
        if nombre.lower().endswith(".xlsx"):
            return filas_xlsx(contenido)
        with zipfile.ZipFile(io.BytesIO(contenido)) as z:
            excels = [n for n in z.namelist() if n.lower().endswith(".xlsx")]
            return sum(filas_xlsx(z.read(n)) for n in excels) if excels else None
    except Exception:
        return None  # archivo que no es Excel legible: se registra sin conteo y se revisa en Silver


def descargar_datatur(categorias=tuple(PAGINAS), contar: bool = True) -> list[dict]:
    """Descarga todos los archivos de las categorías pedidas y los registra en el manifiesto."""
    carpeta_base = BRONZE / "datatur" / date.today().isoformat()
    registros = []
    for categoria in categorias:
        pagina, fuente = PAGINAS[categoria]
        carpeta = carpeta_base / categoria
        carpeta.mkdir(parents=True, exist_ok=True)
        rutas = enlaces_de(pagina)
        print(f"{categoria}: {len(rutas)} archivos")
        for ruta in rutas:
            url = BASE + ruta
            nombre = ruta.rsplit("/", 1)[-1]
            destino = carpeta / nombre
            if not destino.exists():  # permite reanudar si se corta la red
                r = requests.get(url, headers=UA, timeout=300)
                r.raise_for_status()
                destino.write_bytes(r.content)
            filas = contar_filas(destino.read_bytes(), nombre) if contar else None
            registros.append(registrar(fuente, destino, url, filas=filas))
    return registros


In [6]:
%%modulo torre.base.ingesta_abiertas
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Descarga las fuentes abiertas restantes del inventario y las registra en el manifiesto:
#                    D5 Rest-Mex 2025, D6 DENUE (32 estados), D7 Censo 2020 ITER Q. Roo, D8 Open-Meteo (clima
#                    horario y diario de 8 puntos), D9 HURDAT2, D10 FRED, D11 ENDUTIH 2025 (PDF) y D12 GeoJSON.
# Por qué así:       Todas las URL se verificaron el 26–27 de septiembre de 2026 (ver docs/datos/INVENTARIO.md).
#                    Cada fuente es una función corta e independiente: si una falla, las demás siguen, y se
#                    puede volver a correr solo esa.
#                    DENUE se baja completo (32 estados, no solo Q. Roo) porque así la parte de Big Data trabaja con
#                    volumen real y da el contexto nacional de la oferta turística (criterio 2).
# Datos de entrada:  URLs oficiales de INEGI, NOAA, FRED, Open-Meteo, HuggingFace y GitHub.
# Alimenta a:        A1 (oferta y turistas por residente), A3 (clima, huracanes, tipo de cambio) y la campaña
#                    (reseñas y hábitos digitales).

import csv
import io
import json
import time
import zipfile
from datetime import date
from pathlib import Path

import requests

from torre.base.manifiesto import BRONZE, registrar

UA = {"User-Agent": "Mozilla/5.0 (proyecto escolar UNRC)"}
HOY = date.today().isoformat()

# 8 puntos de clima: los 2 destinos emisores y los destinos que se promueven (coordenadas del centro de cada lugar).
PUNTOS_CLIMA = {
    "cancun": (21.1619, -86.8515),
    "playa_del_carmen": (20.6296, -87.0739),
    "tulum": (20.2114, -87.4654),
    "coba": (20.4918, -87.7328),
    "felipe_carrillo_puerto": (19.5801, -88.0452),
    "bacalar": (18.6771, -88.3950),
    "chetumal": (18.5001, -88.2961),
    "kohunlich": (18.4197, -88.7903),
}
ESTADOS_DENUE = [f"{i:02d}" for i in range(1, 33)]


def _bajar(url: str, destino: Path, timeout: int = 300) -> bytes:
    """Descarga una URL a un archivo (salvo que ya exista, para poder reanudar) y devuelve su contenido."""
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists():
        r = requests.get(url, headers=UA, timeout=timeout)
        r.raise_for_status()
        destino.write_bytes(r.content)
    return destino.read_bytes()


def _filas_csv_en_zip(contenido: bytes) -> int:
    """Cuenta las filas de datos (sin encabezado) de los CSV de la carpeta "conjunto_de_datos" dentro de un zip.

    Corrección del 28-sep-2026: la primera versión contaba TODOS los CSV del zip, incluido el diccionario de datos
    (58 líneas en cada zip del DENUE). Eso inflaba el DENUE en 33 × 58 = 1,914 registros (6,139,989 en lugar de
    6,138,075). Se detectó al comparar contra Spark y contra un lector CSV independiente en la Fase 2.
    Segunda corrección (28-sep-2026, Fase 3): la versión anterior solo excluía el diccionario y seguía sumando el
    catálogo "catalogos/tam_loc.csv.csv" del Censo ITER (14 filas): 2,257 en lugar de 2,243 localidades. Ahora se
    cuenta exclusivamente la carpeta conjunto_de_datos, como promete esta descripción.
    """
    total = 0
    with zipfile.ZipFile(io.BytesIO(contenido)) as z:
        for n in z.namelist():
            if n.lower().endswith(".csv") and "conjunto_de_datos" in n.lower():
                with z.open(n) as f:
                    total += max(sum(1 for _ in f) - 1, 0)
    return total


def restmex():
    url = "https://huggingface.co/datasets/vg055/Rest-Mex2025/resolve/main/Rest-Mex_2025_train.csv"
    ruta = BRONZE / "restmex" / HOY / "Rest-Mex_2025_train.csv"
    datos = _bajar(url, ruta)
    # Las reseñas tienen saltos de línea dentro del texto: se cuentan registros con el lector de CSV, no líneas.
    filas = sum(1 for _ in csv.reader(io.StringIO(datos.decode("utf-8", errors="replace")))) - 1
    return [registrar("D5 Rest-Mex 2025", ruta, url, filas=filas, nota="licencia CC-BY-4.0")]


def _es_zip(url: str) -> bool:
    """Pregunta al servidor el tipo de archivo sin descargarlo. INEGI responde una página HTML cuando no existe."""
    r = requests.head(url, headers=UA, timeout=60, allow_redirects=True)
    return r.ok and "zip" in r.headers.get("Content-Type", "")


def denue():
    """DENUE de los 32 estados. Los estados más grandes vienen divididos en partes.

    Verificado el 28-sep-2026: el Estado de México (15) no existe como denue_15_csv.zip, sino como denue_15_1_csv.zip
    (50.6 MB) y denue_15_2_csv.zip (30.2 MB). Por eso, si el archivo único no existe, se buscan las partes _1, _2, …
    """
    base = "https://www.inegi.org.mx/contenidos/masiva/denue/denue_{}_csv.zip"
    salida = []
    for e in ESTADOS_DENUE:
        if _es_zip(base.format(e)):
            nombres = [e]                       # el estado viene en un solo archivo
        else:
            nombres: list[str] = []             # el estado viene en partes: _1, _2, ... hasta que ya no exista
            parte = 1
            while _es_zip(base.format(f"{e}_{parte}")):
                nombres.append(f"{e}_{parte}")
                parte += 1
            if not nombres:
                raise RuntimeError(f"DENUE: no se encontró el estado {e} ni sus partes")
        for n in nombres:
            url = base.format(n)
            ruta = BRONZE / "denue" / HOY / f"denue_{n}_csv.zip"
            datos = _bajar(url, ruta)
            if datos[:2] != b"PK":  # los zip siempre empiezan con "PK"; si no, es una página de error
                ruta.unlink()
                raise RuntimeError(f"DENUE {n}: la descarga no es un zip")
            salida.append(registrar("D6 DENUE INEGI", ruta, url, filas=_filas_csv_en_zip(datos), nota=f"estado {n}"))
    return salida


def iter_qroo():
    url = "https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/iter/iter_23_cpv2020_csv.zip"
    ruta = BRONZE / "iter" / HOY / "iter_23_cpv2020_csv.zip"
    datos = _bajar(url, ruta)
    return [registrar("D7 Censo 2020 ITER", ruta, url, filas=_filas_csv_en_zip(datos))]


def _bajar_con_espera(url: str, ruta: Path, intentos: int = 6) -> bytes:
    """Como _bajar, pero si Open-Meteo responde 429 (demasiadas peticiones) espera y reintenta.

    El 28-sep-2026 la primera versión, que pedía 7.7 años horarios de un golpe, recibió 429 en el segundo punto.
    El servicio gratuito limita cuántos datos se piden por minuto y por hora.
    """
    for i in range(intentos):
        try:
            return _bajar(url, ruta, timeout=600)
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 429 and i < intentos - 1:
                time.sleep(60 * (i + 1))  # 1, 2, 3... minutos
                continue
            raise
    raise RuntimeError(f"Open-Meteo no respondió después de {intentos} intentos: {url}")


def clima():
    """Clima de 8 puntos (reanálisis ERA5 vía Open-Meteo), en partes pequeñas para respetar el límite gratuito:

    - horario (temperatura, lluvia, viento): un archivo por punto y por año, de 2019 a hoy;
    - diario (máximas y acumulados): un archivo por punto y por década, de 1950 a hoy.
    """
    salida, ayer = [], date.fromordinal(date.today().toordinal() - 1)
    tramos = [("horario", f"{a}-01-01", min(date(a, 12, 31), ayer).isoformat(),
               "hourly=temperature_2m,precipitation,wind_speed_10m", str(a)) for a in range(2019, ayer.year + 1)]
    tramos += [("diario", f"{d}-01-01", min(date(d + 9, 12, 31), ayer).isoformat(),
                "daily=temperature_2m_max,precipitation_sum,wind_speed_10m_max&timezone=America%2FCancun", f"{d}s")
               for d in range(1950, ayer.year + 1, 10)]
    for punto, (lat, lon) in PUNTOS_CLIMA.items():
        for tipo, inicio, fin, variables, etiqueta in tramos:
            url = (f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}"
                   f"&start_date={inicio}&end_date={fin}&{variables}")
            ruta = BRONZE / "clima" / HOY / tipo / f"{punto}_{etiqueta}.json"
            ya_estaba = ruta.exists()
            datos = json.loads(_bajar_con_espera(url, ruta))
            serie = datos.get("hourly") or datos.get("daily") or {}
            salida.append(registrar("D8 Open-Meteo", ruta, url, filas=len(serie.get("time", [])), nota=f"{punto} {tipo} {etiqueta}"))
            if not ya_estaba:
                time.sleep(3)  # pausa entre peticiones reales al servidor
    return salida


def huracanes():
    """HURDAT2: se toma el archivo más reciente publicado en el índice de la NOAA."""
    import re
    indice = requests.get("https://www.nhc.noaa.gov/data/hurdat/", headers=UA, timeout=60).text
    nombre = sorted(set(re.findall(r"hurdat2-1851-\d{4}-\d{6}\.txt", indice)))[-1]
    url = f"https://www.nhc.noaa.gov/data/hurdat/{nombre}"
    ruta = BRONZE / "huracanes" / HOY / nombre
    datos = _bajar(url, ruta).decode("utf-8", errors="replace").splitlines()
    tormentas = sum(1 for l in datos if l.startswith("AL"))
    return [registrar("D9 HURDAT2 NOAA", ruta, url, filas=len(datos), nota=f"{tormentas} tormentas")]


def fred():
    salida = []
    for serie in ("DEXMXUS", "CPIAUCSL"):
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={serie}"
        ruta = BRONZE / "fred" / HOY / f"{serie}.csv"
        filas = len(_bajar(url, ruta).decode().splitlines()) - 1
        salida.append(registrar("D10 FRED", ruta, url, filas=filas))
    return salida


def endutih():
    url = "https://www.inegi.org.mx/contenidos/saladeprensa/boletines/2026/endutih/ENDUTIH_25_RR.pdf"
    ruta = BRONZE / "endutih" / HOY / "ENDUTIH_25_RR.pdf"
    _bajar(url, ruta)
    return [registrar("D11 ENDUTIH 2025", ruta, url, nota="reporte de resultados (agregados)")]


def geojson():
    url = "https://raw.githubusercontent.com/PhantomInsights/mexico-geojson/main/2020/states/Quintana%20Roo.json"
    ruta = BRONZE / "geo" / HOY / "quintana_roo.json"
    datos = json.loads(_bajar(url, ruta))
    return [registrar("D12 GeoJSON Q. Roo", ruta, url, filas=len(datos.get("features", [])), nota="polígonos")]


FUENTES = {"restmex": restmex, "denue": denue, "iter": iter_qroo, "clima": clima,
           "huracanes": huracanes, "fred": fred, "endutih": endutih, "geo": geojson}


def descargar_abiertas(fuentes=tuple(FUENTES)):
    """Corre cada fuente por separado; si una falla, se reporta y se sigue con las demás (no se inventa nada)."""
    fallas = {}
    for nombre in fuentes:
        try:
            filas = FUENTES[nombre]()
            print(f"{nombre:10s} OK  {len(filas)} archivo(s), {sum(int(f['filas'] or 0) for f in filas):,} filas")
        except Exception as e:  # se declara el hueco, no se rellena
            fallas[nombre] = repr(e)
            print(f"{nombre:10s} FALLÓ: {e!r}")
    return fallas


In [7]:
%%modulo torre.base.ingesta_benchmarks
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Extrae con un navegador automatizado (Playwright + Chromium) las tablas de costos y
#                    desempeño publicitario por industria de WordStream 2025 y LocaliQ 2026 (Google Ads/búsqueda y
#                    Facebook). Guarda el HTML completo, una captura de pantalla de página completa y todas las tablas
#                    en un CSV largo, y registra todo en el manifiesto como fuente D13.
# Por qué así:       Estas páginas bloquean las descargas automáticas simples (respondieron 403 a una petición
#                    directa el 27-sep-2026), pero muestran sus datos en tablas HTML a cualquier navegador. Un navegador
#                    real las lee igual que una persona y deja captura como evidencia.
#                    Verificado el 28-sep-2026: el CTR de Travel en Google Ads 2025 según la tabla oficial es 8.73 %, no
#                    el 8.24 % de un resumen de búsqueda web. Por eso se extrae de la fuente y no de resúmenes.
#                    Limitación declarada: son promedios de anunciantes de EE. UU.; en el modelo se usan como supuesto
#                    etiquetado, no como medición de México.
# Datos de entrada:  D13 — wordstream.com y localiq.com (páginas públicas de benchmarks).
# Alimenta a:        Fase 6, investigación de operaciones (costo por clic y conversión por canal para repartir el presupuesto).

import csv
import re
from datetime import date

from playwright.sync_api import sync_playwright

from torre.base.manifiesto import BRONZE, registrar

PAGINAS = {
    "wordstream_google_2025": ("WordStream", "Google Ads", 2025, "https://www.wordstream.com/blog/2025-google-ads-benchmarks"),
    "wordstream_facebook_2025": ("WordStream", "Facebook Ads", 2025, "https://www.wordstream.com/blog/facebook-ads-benchmarks-2025"),
    "localiq_busqueda_2026": ("LocaliQ", "Búsqueda (Google/Microsoft)", 2026, "https://localiq.com/blog/search-advertising-benchmarks/"),
    "localiq_facebook_2026": ("LocaliQ", "Facebook Ads", 2026, "https://localiq.com/blog/facebook-advertising-benchmarks/"),
}
NAVEGADOR = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
             "Chrome/131.0 Safari/537.36")


def a_numero(texto: str) -> float | None:
    """'$2.12' → 2.12 · '8.73%' → 8.73 · ' $44.70' → 44.70. Devuelve None si no hay número."""
    m = re.search(r"-?\d+(?:,\d{3})*(?:\.\d+)?", texto.replace("\xa0", " "))
    return float(m.group(0).replace(",", "")) if m else None


def descargar_benchmarks() -> list[dict]:
    carpeta = BRONZE / "benchmarks" / date.today().isoformat()
    carpeta.mkdir(parents=True, exist_ok=True)
    filas_csv, registros = [], []
    with sync_playwright() as p:
        navegador = p.chromium.launch()
        pagina = navegador.new_page(user_agent=NAVEGADOR, viewport={"width": 1366, "height": 900})
        for clave, (editor, canal, anio, url) in PAGINAS.items():
            pagina.goto(url, timeout=90_000, wait_until="domcontentloaded")
            pagina.wait_for_timeout(4_000)  # deja que termine de pintar las tablas

            html = carpeta / f"{clave}.html"
            html.write_text(pagina.content(), encoding="utf-8")
            captura = carpeta / f"{clave}.png"
            pagina.screenshot(path=str(captura), full_page=True)

            n_tablas = pagina.locator("table").count()
            for i in range(n_tablas):
                lineas = [l.split("\t") for l in pagina.locator("table").nth(i).inner_text().splitlines() if l.strip()]
                encabezado, cuerpo = lineas[0], lineas[1:]
                metrica = encabezado[1].strip() if len(encabezado) > 1 else f"tabla_{i}"
                for celdas in cuerpo:
                    if len(celdas) < 2:
                        continue
                    filas_csv.append({
                        "clave_pagina": clave, "editor": editor, "canal": canal, "anio": anio,
                        "tabla": i, "metrica": metrica, "categoria": celdas[0].strip(),
                        "valor_texto": celdas[1].strip(), "valor": a_numero(celdas[1]), "url": url,
                    })
            registros.append(registrar("D13 Benchmarks", html, url, filas=n_tablas, nota=f"{editor} {canal} {anio}: HTML ({n_tablas} tablas)"))
            registros.append(registrar("D13 Benchmarks", captura, url, nota="captura de página completa (evidencia)"))
        navegador.close()

    salida = carpeta / "benchmarks_tablas.csv"
    with open(salida, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(filas_csv[0]))
        w.writeheader()
        w.writerows(filas_csv)
    registros.append(registrar("D13 Benchmarks", salida, "extraído de las 4 páginas con Playwright", filas=len(filas_csv),
                               nota="todas las tablas, todas las industrias"))
    return filas_csv


## 3. Spark
Spark reparte el trabajo entre los núcleos de la computadora. Corre en **modo local** (`local[*]`): el mismo programa
correría en un clúster de muchas computadoras cambiando solo la dirección del maestro. `crear_spark` (en
`torre.base.entorno`) apunta Java 17 y Hadoop solo dentro de este proceso.

### El código de la sesión de Spark
Cada celda de código de abajo es un archivo del paquete `torre`, completo y sin cambios (con su encabezado: qué hace, por qué así, de qué datos sale y a qué decisión alimenta). Al correrla, la celda queda registrada como ese módulo y las celdas siguientes lo usan.

- **`torre.base.entorno`** — Prepara el entorno para que PySpark funcione en Windows y crea la sesión de Spark.

In [8]:
%%modulo torre.base.entorno
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Prepara el entorno para que PySpark funcione en Windows y crea la sesión de Spark.
# Por qué así:       En esta máquina el Java por defecto es 1.8 (verificado el 26-sep-2026) y PySpark necesita
#                    Java 17. En lugar de cambiar el Java de todo Windows, este módulo apunta JAVA_HOME al JDK 17
#                    solo para el proceso del proyecto. HADOOP_HOME apunta a herramientas/hadoop (winutils.exe y
#                    hadoop.dll 3.3.6), sin los cuales Spark no puede escribir Parquet en Windows.
#                    Alternativa descartada: correr Spark en Docker (funciona, pero es más pesado de explicar y de
#                    arrancar el día del coloquio).
# Datos de entrada:  Ninguno (solo configura el entorno).
# Alimenta a:        Todas las tuberías de Bronze → Silver → Gold (criterio 2 de la rúbrica).

import glob
import os
import sys
from pathlib import Path

# Raíz del proyecto: este archivo vive en backend/torre/base/, así que subimos tres niveles.
RAIZ = Path(__file__).resolve().parents[3]
HADOOP_HOME = RAIZ / "herramientas" / "hadoop"


def buscar_jdk17() -> Path:
    """Busca un JDK 17 instalado en las rutas estándar de Windows.

    Devuelve la carpeta del JDK. Si no hay uno, lanza un error que dice exactamente
    cómo instalarlo; no se sigue con otro Java porque PySpark fallaría más adelante.
    """
    candidatos = []
    for patron in (r"C:\Program Files\Microsoft\jdk-17*",
                   r"C:\Program Files\Eclipse Adoptium\jdk-17*",
                   r"C:\Program Files\Java\jdk-17*"):
        candidatos.extend(glob.glob(patron))
    if not candidatos:
        raise RuntimeError(
            "No encontré un JDK 17. Instálalo con:\n"
            "    winget install Microsoft.OpenJDK.17\n"
            "y vuelve a correr la prueba."
        )
    # Si hay varios, se usa el de versión más alta. Como texto, "jdk-17.0.9" quedaría después de
    # "jdk-17.0.20", así que se ordena por los números de la versión y no alfabéticamente.
    def version(ruta: str) -> tuple:
        numeros = "".join(c if c.isdigit() else " " for c in Path(ruta).name).split()
        return tuple(int(n) for n in numeros)
    return Path(max(candidatos, key=version))


def configurar_entorno() -> dict:
    """Fija JAVA_HOME, HADOOP_HOME y PATH solo para este proceso. Devuelve lo que configuró."""
    jdk = buscar_jdk17()
    os.environ["JAVA_HOME"] = str(jdk)
    os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
    # Se antepone al PATH para que gane sobre el Java 8 del sistema.
    os.environ["PATH"] = os.pathsep.join([str(jdk / "bin"), str(HADOOP_HOME / "bin"), os.environ["PATH"]])
    # Spark arranca "trabajadores" de Python cuando recibe datos creados en Python. En Windows, si no se le dice qué
    # Python usar, se queda esperando y falla con "Accept timed out" (visto el 28-sep-2026 al construir la ocupación
    # de DataTur). Se le indica el mismo Python del proyecto (.venv).
    python = _ruta_corta(sys.executable)
    os.environ["PYSPARK_PYTHON"] = python
    os.environ["PYSPARK_DRIVER_PYTHON"] = python
    return {"JAVA_HOME": str(jdk), "HADOOP_HOME": str(HADOOP_HOME), "PYSPARK_PYTHON": python}


def _ruta_corta(ruta: str) -> str:
    """Devuelve la ruta "corta" de Windows, sin espacios (por ejemplo, la carpeta "PP 5to semestre..." pasa a "PP5TOS~1").

    La carpeta del proyecto tiene espacios ("PP 5to semestre ...") y los scripts de Spark la cortan en "5to"
    ("'5to' is not recognized", visto el 28-sep-2026). La ruta corta evita el problema. Fuera de Windows no cambia nada.
    """
    if os.name != "nt":
        return ruta
    import ctypes

    buffer = ctypes.create_unicode_buffer(1024)
    n = ctypes.windll.kernel32.GetShortPathNameW(ruta, buffer, len(buffer))
    return buffer.value if 0 < n < len(buffer) else ruta


def crear_spark(nombre_app: str = "torre-del-caribe"):
    """Crea una sesión de Spark local que usa todos los núcleos de la máquina.

    local[*]    = modo local con todos los núcleos (12 en esta máquina).
    driver 4g   = memoria para el proceso principal (la máquina tiene 16 GB).
    timeZone    = UTC, para que las fechas no se muevan entre máquinas.
    progreso    = apagado, para que los resúmenes se lean limpios.
    """
    configurar_entorno()
    from pyspark.sql import SparkSession  # se importa después de fijar JAVA_HOME

    return (
        SparkSession.builder.appName(nombre_app)
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.session.timeZone", "UTC")
        # Sin barra de progreso en la terminal: tapaba los resúmenes impresos por los procesos.
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )


In [9]:
from torre.base.entorno import crear_spark
spark = crear_spark("entrega-datos")
print("Spark", spark.version, "· núcleos:", spark.sparkContext.defaultParallelism)
BRONZE_COMPLETO = (RAIZ / "datos" / "bronze" / "denue").is_dir()
print("¿Está el crudo completo para volver a limpiar?", "sí" if BRONZE_COMPLETO else "no: se usan las tablas limpias que ya vienen")

Spark 3.5.6 · núcleos: 12
¿Está el crudo completo para volver a limpiar? sí


## 4. La limpieza, una fuente a la vez (de bronce a plata)
Un archivo por fuente, porque cada una tiene sus propias trampas: ceros que en realidad son huecos, renglones repetidos,
notas al pie que parecen datos, aerolíneas con la misma etiqueta. Cada regla está escrita en el encabezado del módulo y
se comprobó con los datos.

### El código de la limpieza
Cada celda de código de abajo es un archivo del paquete `torre`, completo y sin cambios (con su encabezado: qué hace, por qué así, de qué datos sale y a qué decisión alimenta). Al correrla, la celda queda registrada como ese módulo y las celdas siguientes lo usan.

- **`torre.base.silver_siturq`** — Convierte las respuestas crudas de SITUR-Q (Bronze, JSON anidado) en una tabla larga y uniforme en Silver: indicador, unidad (destino o zona), año, mes, variable, valor y banderas de calidad. Se procesa con PySpark y se guarda en Parquet, particionado por indicador.
- **`torre.base.silver_datatur_ocupacion`** — Convierte los 166 Excel de ocupación hotelera de DataTur (135 semanales y 31 mensuales) en una tabla limpia en Silver: centro turístico, año, semana o mes, cuartos disponibles, cuartos ocupados y porcentaje de ocupación, con una sola versión por periodo.
- **`torre.base.silver_denue`** — Procesa con PySpark los 6.1 millones de negocios del DENUE (INEGI, 32 estados) y los guarda en Silver: giro (código SCIAN), tamaño (personas ocupadas), ubicación (estado, municipio, localidad, latitud y longitud) y si es oferta turística. Se particiona por estado.
- **`torre.base.silver_inah`** — Limpia la base de visitantes del INAH (DataTur, BdINAH: museos y zonas arqueológicas de todo el país, 2016–2026) y la guarda en Silver con una fila por sitio, mes y tipo de visitante (nacional o extranjero). A las zonas de Quintana Roo les agrega su papel en la campaña: promovida (una de las 5 regiones), referencia, retirada o excluida.
- **`torre.base.silver_iter`** — Limpia el Censo de Población y Vivienda 2020 de Quintana Roo por localidad (INEGI, ITER) y lo guarda en Silver: población, viviendas habitadas, servicios de las viviendas (agua entubada, drenaje, electricidad) y coordenadas en grados decimales. A cada localidad le pone la región de la campaña a la que pertenece (o "referencia"), para calcular turistas por residente.
- **`torre.base.silver_clima`** — Limpia el clima de Open-Meteo (reanálisis ERA5) de 8 puntos de Quintana Roo y lo guarda en Silver en dos tablas: clima_diario (1950–2026: temperatura máxima, lluvia del día y viento máximo) y clima_horario (2019–2026: temperatura, lluvia y viento de cada hora).
- **`torre.base.silver_huracanes`** — Limpia HURDAT2 (NOAA, todas las tormentas del Atlántico 1851–2025) y la guarda en Silver con una fila por punto de trayectoria (cada 6 horas). A cada punto le calcula su distancia a Chetumal y le pone las banderas que definen "una tormenta que afecta al sur de Quintana Roo".
- **`torre.base.silver_fred`** — Limpia dos series de FRED (Reserva Federal de St. Louis) y las guarda en Silver: fred_diario  → tipo de cambio pesos por dólar (DEXMXUS), un dato por día hábil desde 1993; fred_mensual → promedio mensual del tipo de cambio y el índice de precios al consumidor de EE. UU. (CPIAUCSL), una fila por mes.
- **`torre.base.silver_nacionalidad`** — Limpia la base de llegadas de extranjeros por avión según su nacionalidad (DataTur, BD_Nacionalidad, con cifras de la Unidad de Política Migratoria) y la guarda en Silver con una fila por mes, aeropuerto, país y sexo. A los aeropuertos de Quintana Roo les pone su lugar y su papel en la campaña.
- **`torre.base.silver_afac`** — Limpia la base de pasajeros por aerolínea de la AFAC (DataTur, DB_AFAC) y la guarda en Silver con una fila por mes, tipo de vuelo, servicio, región de la aerolínea y aerolínea.
- **`torre.base.silver_cruceros`** — Limpia la base de cruceros por puerto (DataTur, BaseDatosCruceros: arribos y pasajeros por mes, 2016 → jul-2026) y la guarda en Silver. Después la compara con los cruceristas de SITUR-Q (la otra fuente del mismo dato) y guarda la diferencia en Gold.
- **`torre.base.silver_restmex`** — Limpia el corpus de reseñas Rest-Mex 2025 (208,051 reseñas de turistas en México, con calificación de 1 a 5, pueblo, estado y tipo: hotel, restaurante o atractivo) y lo guarda en Silver. PySpark lee el CSV directo (el texto trae saltos de línea dentro de las comillas).

In [10]:
%%modulo torre.base.silver_siturq
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Convierte las respuestas crudas de SITUR-Q (Bronze, JSON anidado) en una tabla larga y uniforme en
#                    Silver: indicador, unidad (destino o zona), año, mes, variable, valor y banderas de calidad.
#                    Se procesa con PySpark y se guarda en Parquet, particionado por indicador.
# Por qué así:       Cada indicador de SITUR-Q trae campos distintos ("Ocupación hotelera", "Total", "Cruceristas"...).
#                    Una tabla larga (una fila por destino-mes-variable) permite tratarlos todos igual en los modelos.
#                    Reglas aplicadas, cada una verificada con los datos el 28-sep-2026:
#                    1. Tren Maya: algunos destinos traen 2 filas por mes (una por estación). Se SUMAN, porque la suma
#                       cuadra exacto con el total de zona que publica SITUR-Q. Ejemplo, descensos de ene-2025:
#                       Grand Costa Maya 7,084 = Chetumal (24 + 3,478) + Bacalar (213 + 3,369).
#                    2. Ocupación hotelera: si "habitaciones disponibles" vale 0, el mes es un HUECO (nulo con bandera).
#                       Que no haya cuartos disponibles es imposible: el indicador de habitaciones sí reporta cuartos en
#                       2025. Esto afecta a todos los destinos en 2025.
#                    3. Afluencia de turistas y derrama: un 0 también es imposible (un destino no recibe cero turistas ni
#                       cero pesos en un mes); son casillas vacías y se marcan como HUECO. Corrección del 28-sep-2026: la
#                       primera versión los conservaba y hacía parecer que había datos hasta abril y junio de 2024, cuando
#                       en realidad terminan en marzo de 2024.
#                    4. En los demás indicadores el 0 se conserva, porque puede ser real (cruceristas de 2020, con los
#                       puertos cerrados por la pandemia; zonas arqueológicas cerradas).
#                    5. Las consultas que fallaron en la fuente (respuesta nula) no generan filas: son huecos declarados
#                       en Bronze.
#                    6. Llegadas en avión: un mes en que todos los aeropuertos marcan 0 a la vez es hueco (2025–2026);
#                       un 0 aislado se conserva (Tulum antes de abrir, dic-2023).
#                    Alternativa descartada: rellenar huecos con promedios. La regla de oro 1 lo prohíbe.
# Datos de entrada:  datos/bronze/siturq/<fecha>/*.json (D1).
# Alimenta a:        Gold (hechos por destino y mes), A1 Radar y A3 Pronóstico.

import re
import unicodedata
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from torre.base.entorno import RAIZ, crear_spark

BRONZE_SITURQ = RAIZ / "datos" / "bronze" / "siturq"
SILVER_SITURQ = RAIZ / "datos" / "silver" / "siturq"
# Campos que describen la fila (no son medidas) y se descartan al pasar a formato largo.
NO_MEDIDAS = {"Año", "Mes", "Destino", "Nombre de destino", "Activo", "Fecha de alta", "Fecha de modificación"}


def a_snake(texto: str) -> str:
    """'Ocupación hotelera' → 'ocupacion_hotelera' (sin acentos, minúsculas, guiones bajos)."""
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", sin_acentos.lower()).strip("_")


def leer_indicador(spark: SparkSession, archivo: Path) -> DataFrame:
    """Lee un JSON de Bronze y lo deja en formato largo: una fila por unidad-año-mes-variable."""
    crudo = spark.read.option("multiLine", True).json(str(archivo))
    # Si TODAS las consultas de un indicador fallaron (respuesta nula), Spark no puede inferir la estructura y la lee
    # como texto. Pasa con "turista_afluencia" (120 de 120 errores 500 en la fuente): ese indicador no aporta filas.
    if "respuesta" not in crudo.columns or not isinstance(crudo.schema["respuesta"].dataType, T.StructType):
        return None
    # Cada registro trae una lista "data" con las filas mensuales; se "desenrolla" (explode) en filas.
    filas = (crudo.where(F.col("respuesta").isNotNull())
             .select("indicador", "unidad", "tipo_unidad", "anio", F.explode_outer("respuesta.data").alias("f"))
             .where(F.col("f").isNotNull())
             .select("indicador", "unidad", "tipo_unidad", "anio", "f.*"))
    medidas = [c for c in filas.columns if c not in NO_MEDIDAS | {"indicador", "unidad", "tipo_unidad", "anio"}]
    if not medidas:
        return None
    # stack() convierte columnas en filas: (variable, valor). El valor llega como texto ("66.50") o número.
    pares = ", ".join(f"'{a_snake(m)}', cast(`{m}` as double)" for m in medidas)
    return (filas.select("indicador", "unidad", "tipo_unidad", "anio", F.col("Mes").cast("int").alias("mes"),
                         F.expr(f"stack({len(medidas)}, {pares}) as (variable, valor)")))


def construir_silver_siturq(spark: SparkSession | None = None) -> DataFrame:
    """Une todos los indicadores de la descarga más reciente, aplica las reglas y guarda Silver en Parquet."""
    spark = spark or crear_spark("silver-siturq")
    carpeta = sorted(p for p in BRONZE_SITURQ.iterdir() if p.is_dir())[-1]  # descarga más reciente
    partes = [df for a in sorted(carpeta.glob("*.json")) if (df := leer_indicador(spark, a)) is not None]
    largo = partes[0]
    for df in partes[1:]:
        largo = largo.unionByName(df)

    # Regla 1: sumar filas repetidas del mismo destino-mes-variable (estaciones del Tren Maya).
    agregado = (largo.groupBy("indicador", "unidad", "tipo_unidad", "anio", "mes", "variable")
                .agg(F.sum("valor").alias("valor"), F.count("*").alias("n_registros_fuente")))

    # Regla 2: meses de ocupación con 0 habitaciones disponibles = hueco.
    sin_cuartos = (agregado.where((F.col("indicador") == "ocupacion_hotelera")
                                  & (F.col("variable") == "numero_de_habitaciones_disponibles") & (F.col("valor") == 0))
                   .select("indicador", "unidad", "anio", "mes").withColumn("hueco_flag", F.lit(True)))
    # Regla 3: en afluencia y derrama, el mes completo en 0 es una casilla vacía.
    flujos = ["afluencia_turistas", "derrama_turistas", "derrama_visitantes"]
    flujo_cero = (agregado.where(F.col("indicador").isin(flujos)).groupBy("indicador", "unidad", "anio", "mes")
                  .agg(F.max("valor").alias("maximo")).where("maximo = 0")
                  .select("indicador", "unidad", "anio", "mes").withColumn("hueco_flag", F.lit(True)))
    # Regla 6 (28-sep-2026): llegadas en avión. Si en un mes TODOS los aeropuertos reportan 0 a la vez, la fuente dejó de
    # publicar (así ocurre en todo 2025 y 2026: Cancún no recibió cero pasajeros) y el mes es hueco. Un 0 aislado se
    # conserva: el aeropuerto de Tulum abrió en diciembre de 2023, así que sus ceros de ese año son reales.
    aereos_cero = (agregado.where(F.col("indicador") == "aereos_llegadas").groupBy("anio", "mes")
                   .agg(F.max("valor").alias("maximo")).where("maximo = 0").select("anio", "mes"))
    aereos_hueco = (agregado.where(F.col("indicador") == "aereos_llegadas").join(aereos_cero, ["anio", "mes"])
                    .select("indicador", "unidad", "anio", "mes").distinct().withColumn("hueco_flag", F.lit(True)))
    sin_cuartos = sin_cuartos.unionByName(flujo_cero).unionByName(aereos_hueco)
    silver = (agregado.join(sin_cuartos, ["indicador", "unidad", "anio", "mes"], "left")
              .withColumn("hueco_flag", F.coalesce("hueco_flag", F.lit(False)))
              .withColumn("valor", F.when(F.col("hueco_flag"), F.lit(None).cast("double")).otherwise(F.col("valor")))
              .withColumn("periodo", F.make_date("anio", "mes", F.lit(1)))
              .withColumn("fuente", F.lit(f"SITUR-Q, descarga {carpeta.name}")))

    silver.write.mode("overwrite").partitionBy("indicador").parquet(str(SILVER_SITURQ))
    return spark.read.parquet(str(SILVER_SITURQ))


In [11]:
%%modulo torre.base.silver_datatur_ocupacion
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Convierte los 166 Excel de ocupación hotelera de DataTur (135 semanales y 31 mensuales) en una
#                    tabla limpia en Silver: centro turístico, año, semana o mes, cuartos disponibles, cuartos
#                    ocupados y porcentaje de ocupación, con una sola versión por periodo.
# Por qué así:       Cada archivo compara TRES años del mismo periodo: el de la semana 31 de 2026 trae la semana 31 de
#                    2024, 2025 y 2026; el de 2024 trae 2022, 2023 y 2024. Por eso la serie semanal va de 2022 a 2026
#                    (verificado el 28-sep-2026) y cada periodo aparece en varios archivos. DataTur marca las cifras
#                    como preliminares y las corrige después, así que se conserva la versión del ARCHIVO MÁS
#                    RECIENTE y se cuenta cuántas veces cambió (revisado_flag).
#                    Los Excel se leen con openpyxl porque Spark no lee Excel de forma nativa. Son archivos pequeños;
#                    la unión, la deduplicación y la escritura en Parquet las hace Spark.
#                    Se conservan los 54 centros del país (contexto nacional) y se marca es_qroo. Las filas de
#                    subtotal ("Total", "Centros de Playa"...) se marcan tipo_fila = "agregado".
#                    Las celdas "n.d." (no disponible) y "n.c." (no comparable) se guardan como nulo con bandera.
# Datos de entrada:  datos/bronze/datatur/<fecha>/hoteleria/*.zip (D2 y D2m).
# Alimenta a:        A1 Radar (ocupación semanal del norte), A3 Pronóstico y la reconciliación con SITUR-Q.

import io
import re
import zipfile
from datetime import date
from pathlib import Path

import openpyxl
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

from torre.base.entorno import RAIZ, crear_spark

BRONZE_HOTELERIA = RAIZ / "datos" / "bronze" / "datatur"
SILVER = RAIZ / "datos" / "silver" / "datatur_ocupacion"
MESES = {m: i for i, m in enumerate(["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
                                     "septiembre", "octubre", "noviembre", "diciembre"], start=1)}
# Columnas del Excel (misma posición en los 166 archivos): años en 2-4, 9-11 y 16-18.
AGREGADOS = {"Total", "Centros De Playa", "Integralmente Planeados", "Tradicionales", "Otros", "Ciudades",
             "Grandes", "Del Interior", "Fronterizas"}
BLOQUES = {"cuartos_disponibles": (2, 3, 4), "cuartos_ocupados": (9, 10, 11), "ocupacion_pct": (16, 17, 18)}


def _numero(celda):
    """Número o None. 'n.d.' / 'n.c.' / vacío → None."""
    return float(celda) if isinstance(celda, (int, float)) else None


def leer_archivo(ruta: Path) -> list[dict]:
    """Lee un zip de DataTur (semanal o mensual) y devuelve una fila por centro y año."""
    with zipfile.ZipFile(ruta) as z:
        nombre_xlsx = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
        libro = openpyxl.load_workbook(io.BytesIO(z.read(nombre_xlsx)), read_only=True, data_only=True)
    hoja = libro.worksheets[0]  # la primera hoja es el periodo; la segunda, el acumulado del año (no se usa)
    filas = list(hoja.iter_rows(values_only=True))
    libro.close()

    titulo = str(filas[1][2])
    anios = [int(re.sub(r"\D", "", str(filas[6][c]))) for c in BLOQUES["cuartos_disponibles"]]
    semanal = "Semana" in titulo
    periodo = int(re.search(r"Semana\s+(\d+)", titulo).group(1)) if semanal else \
        MESES[re.search(r"Mes de (\w+)", titulo).group(1).lower()]
    # Orden del archivo para decidir cuál es "el más reciente": (año del archivo, número de semana o mes).
    orden = (anios[-1], periodo)

    # Notas al pie del archivo ("4/ A partir del 1 de septiembre 2025 se actualizó la oferta hotelera..."). Los centros
    # las referencian con marcas como "ISLA MUJERES, Q. ROO /4". Se guardan porque indican cortes de comparabilidad.
    notas = {}
    for fila in filas[9:]:
        texto = str(fila[1] or "").strip()
        m = re.match(r"^(\d+)/\s*(.+)", texto)
        if m and not any(isinstance(c, (int, float)) for c in fila[2:19]):
            notas[m.group(1)] = m.group(2).strip()

    salida = []
    for fila in filas[9:]:
        centro_crudo = str(fila[1] or "").strip()
        # Solo son filas de datos las que traen algún número; así se descartan notas, fechas y la línea de "Fuente".
        if not centro_crudo or not any(isinstance(c, (int, float)) for c in fila[2:19]):
            continue
        es_centro = "," in centro_crudo
        nombre, _, estado = centro_crudo.partition(",")
        marca = re.search(r"/(\d+)", centro_crudo)
        # El estado se escribe distinto entre archivos ("Q.ROO", "Q. ROO", "B.C.S.1/", "BC", "B.C."): se quitan espacios,
        # puntos y marcas de nota, y queda la abreviatura limpia ("QROO", "BCS", "BC").
        estado = re.sub(r"[\s.]|/?\d+/?", "", estado).upper()
        if len(estado) > 5:  # no es una abreviatura de estado sino una nota al pie con comas y números: se descarta
            continue
        # Limpia marcas de nota ("/3", " 2/") y puntos finales: "Ciudad De México 2/" → "Ciudad De México".
        nombre = re.sub(r"\s*(/\d+|\d+/)\s*$", "", nombre).strip().rstrip(".").title()
        tipo = "centro" if es_centro else ("agregado" if nombre in AGREGADOS else "desglose")
        for i, anio in enumerate(anios):
            valores = {var: fila[cols[i]] for var, cols in BLOQUES.items()}
            salida.append({
                "frecuencia": "semanal" if semanal else "mensual",
                "centro": nombre, "estado": estado or None, "centro_crudo": centro_crudo, "tipo_fila": tipo,
                "es_qroo": estado == "QROO",
                "anio": anio, "semana": periodo if semanal else None, "mes": None if semanal else periodo,
                # Fecha de inicio del periodo: lunes de la semana ISO (así numera DataTur; la semana 31 de 2026 empieza
                # el 27-jul-2026, igual que dice el título del archivo) o el día 1 del mes.
                "periodo": date.fromisocalendar(anio, periodo, 1) if semanal else date(anio, periodo, 1),
                **{var: _numero(v) for var, v in valores.items()},
                "no_disponible_flag": any(isinstance(v, str) for v in valores.values()),
                "nota_al_pie": notas.get(marca.group(1)) if marca else None,
                "archivo": ruta.name, "orden_archivo": orden[0] * 100 + orden[1],
            })
    return salida


def construir_silver_ocupacion(spark: SparkSession | None = None):
    spark = spark or crear_spark("silver-datatur-ocupacion")
    carpeta = sorted(p for p in BRONZE_HOTELERIA.iterdir() if p.is_dir())[-1] / "hoteleria"
    registros = [r for ruta in sorted(carpeta.glob("*.zip")) for r in leer_archivo(ruta)]
    df = spark.createDataFrame(registros)

    # Una versión por centro-año-periodo: la del archivo más reciente. También se cuenta si cambió entre versiones.
    # La llave usa el nombre LIMPIO: DataTur cambia las marcas de nota al pie entre archivos ("ISLA MUJERES, Q. ROO /4"
    # en uno y sin "/4" en otro). Con el nombre crudo, una misma semana se contaba dos veces (visto el 28-sep-2026).
    llave = ["frecuencia", "centro", "estado", "tipo_fila", "anio", "semana", "mes"]
    w = Window.partitionBy(*llave).orderBy(F.col("orden_archivo").desc())
    versiones = (df.withColumn("rn", F.row_number().over(w))
                 .withColumn("n_versiones", F.count("*").over(Window.partitionBy(*llave)))
                 .withColumn("ocup_distintas", F.size(F.collect_set("ocupacion_pct").over(Window.partitionBy(*llave)))))
    silver = (versiones.where("rn = 1").drop("rn", "orden_archivo")
              .withColumn("revisado_flag", F.col("ocup_distintas") > 1).drop("ocup_distintas")
              # Comprobación interna: ocupación recalculada = ocupados / disponibles.
              .withColumn("ocupacion_calc_pct", F.round(F.col("cuartos_ocupados") / F.col("cuartos_disponibles") * 100, 2)))
    silver.write.mode("overwrite").partitionBy("frecuencia").parquet(str(SILVER))
    return spark.read.parquet(str(SILVER))


In [12]:
%%modulo torre.base.silver_denue
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Procesa con PySpark los 6.1 millones de negocios del DENUE (INEGI, 32 estados) y los guarda en
#                    Silver: giro (código SCIAN), tamaño (personas ocupadas), ubicación (estado, municipio, localidad,
#                    latitud y longitud) y si es oferta turística. Se particiona por estado.
# Por qué así:       - Volumen real: es la tabla más grande del proyecto y sostiene el criterio de Big Data. Brandon
#                      decidió procesarla completa (28-sep-2026) para comparar Quintana Roo contra el país.
#                    - Privacidad (incidente de Big Data): se DESCARTAN razón social, teléfono y correo. La razón social
#                      puede ser el nombre de una persona física (la primera fila del archivo de Q. Roo lo es). Para la
#                      campaña bastan el giro, el tamaño y la ubicación.
#                    - Oferta turística = giros característicos del turismo según SECTUR/INEGI (decisión de Brandon):
#                      721 alojamiento, 722 alimentos y bebidas, 5615 agencias de viajes, 487 transporte turístico,
#                      712 museos y sitios históricos, 713 esparcimiento.
#                    - Los archivos vienen en latin-1 (verificado el 28-sep-2026), no en UTF-8.
#                    - Spark no lee zips: los CSV se descomprimen en una carpeta temporal que se borra al final. Bronze
#                      no se toca.
# Datos de entrada:  datos/bronze/denue/<fecha>/denue_*_csv.zip (D6; 33 archivos, el Estado de México en 2 partes).
# Alimenta a:        A1 Radar (densidad de oferta turística por destino), la selección de regiones (Fase 3) y la campaña
#                    (qué servicios hay para el visitante en cada lugar).

import shutil
import zipfile
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from torre.base.entorno import RAIZ, crear_spark

BRONZE_DENUE = RAIZ / "datos" / "bronze" / "denue"
SILVER_DENUE = RAIZ / "datos" / "silver" / "denue"
TEMPORAL = RAIZ / "datos" / "silver" / "_temporal_denue"

# Prefijo SCIAN → categoría turística. Se revisa primero el prefijo de 4 dígitos (5615) y luego los de 3.
CATEGORIAS_TURISTICAS = {
    "721": "Alojamiento", "722": "Alimentos y bebidas", "5615": "Agencias de viajes",
    "487": "Transporte turístico", "712": "Museos y sitios históricos", "713": "Esparcimiento",
}
COLUMNAS = ["id", "nom_estab", "codigo_act", "nombre_act", "per_ocu", "tipoUniEco", "cve_ent", "entidad", "cve_mun",
            "municipio", "cve_loc", "localidad", "cod_postal", "latitud", "longitud", "fecha_alta"]


def descomprimir(carpeta: Path) -> Path:
    """Extrae solo el CSV de datos de cada zip a la carpeta temporal (se omiten diccionario y metadatos)."""
    if TEMPORAL.exists():
        shutil.rmtree(TEMPORAL)
    TEMPORAL.mkdir(parents=True)
    for z in sorted(carpeta.glob("denue_*_csv.zip")):
        with zipfile.ZipFile(z) as arch:
            for n in arch.namelist():
                if n.startswith("conjunto_de_datos/") and n.lower().endswith(".csv"):
                    (TEMPORAL / Path(n).name).write_bytes(arch.read(n))
    return TEMPORAL


def construir_silver_denue(spark: SparkSession | None = None):
    spark = spark or crear_spark("silver-denue")
    carpeta = sorted(p for p in BRONZE_DENUE.iterdir() if p.is_dir())[-1]
    temporal = descomprimir(carpeta)
    try:
        crudo = (spark.read.option("header", True).option("encoding", "ISO-8859-1")
                 .option("quote", '"').option("escape", '"').csv(str(temporal / "*.csv")))
        n_crudo = crudo.count()

        scian = F.col("codigo_act")
        categoria = F.when(scian.startswith("5615"), CATEGORIAS_TURISTICAS["5615"])
        for prefijo in ("721", "722", "487", "712", "713"):
            categoria = categoria.when(scian.startswith(prefijo), CATEGORIAS_TURISTICAS[prefijo])

        # "11 a 30 personas" → mínimo 11 y máximo 30; "251 y más personas" → mínimo 251, sin máximo.
        rango = F.regexp_extract_all(F.col("per_ocu"), F.lit(r"(\d+)"), 1)
        lat, lon = F.col("latitud").cast("double"), F.col("longitud").cast("double")

        silver = (crudo.select(*COLUMNAS)
                  .dropDuplicates(["id"])                                       # un negocio, una fila
                  .withColumn("localidad", F.trim("localidad"))
                  .withColumn("scian_3", F.substring("codigo_act", 1, 3))
                  .withColumn("categoria_turistica", categoria)
                  .withColumn("es_turistico", F.col("categoria_turistica").isNotNull())
                  .withColumn("personas_min", rango.getItem(0).cast("int"))
                  .withColumn("personas_max", rango.getItem(1).cast("int"))
                  .withColumn("latitud", lat).withColumn("longitud", lon)
                  # Bandera de calidad: coordenadas vacías o fuera de México (latitud 14–33 N, longitud 86–119 O).
                  .withColumn("coordenadas_flag", ~(lat.between(14, 33) & lon.between(-119, -86)) | lat.isNull())
                  .withColumn("es_qroo", F.col("cve_ent") == "23")
                  .withColumn("fuente", F.lit(f"INEGI DENUE, descarga {carpeta.name}")))
        silver.write.mode("overwrite").partitionBy("cve_ent").parquet(str(SILVER_DENUE))
    finally:
        shutil.rmtree(temporal, ignore_errors=True)  # la carpeta temporal nunca se queda en disco
    resultado = spark.read.parquet(str(SILVER_DENUE))
    return resultado, n_crudo


In [13]:
%%modulo torre.base.silver_inah
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Limpia la base de visitantes del INAH (DataTur, BdINAH: museos y zonas arqueológicas de todo el
#                    país, 2016–2026) y la guarda en Silver con una fila por sitio, mes y tipo de visitante (nacional o
#                    extranjero). A las zonas de Quintana Roo les agrega su papel en la campaña: promovida (una de las
#                    5 regiones), referencia, retirada o excluida.
# Por qué así:       - Es la fuente medida de presión turística que sí llega a 2026 en el sur (SITUR-Q no tiene
#                      ocupación hotelera 2025–2026). Sostiene la tabla de criterios de la Fase 3.
#                    - Duplicado detectado el 28-sep-2026: el bloque "Extranjero, septiembre de 2025" viene dos veces
#                      (283 sitios × 2). Una copia trae las cifras y la otra puros ceros. Se conserva la fila con cifra.
#                      Si alguna vez aparecen dos cifras distintas de cero para la misma llave, el proceso se detiene
#                      (regla de oro 5): no se elige una al azar.
#                    - El papel de cada zona sale de docs/regiones/REGIONES.md (D.5, 28-sep-2026). Alternativa
#                      descartada: guardar solo Quintana Roo; se conserva el país para comparar (criterio 2).
#                    - Los ceros se conservan: un mes con 0 visitantes suele ser un cierre (Muyil, jun-2024 a feb-2026)
#                      y se marca con sin_visitantes_flag, no se borra.
#                    - Excel → pandas (Spark no lee Excel) y Spark escribe el Parquet particionado, igual que DataTur.
# Datos de entrada:  datos/bronze/datatur/<fecha>/inah/BdINAH.zip (D4).
# Alimenta a:        Fase 3 (tabla de criterios de las 5 regiones), A1 Radar (visitas INAH como variable de presión) y
#                    la campaña (qué sitios de la ruta sur crecen en 2026).

import io
import zipfile
from pathlib import Path

import pandas as pd

from torre.base.entorno import RAIZ, crear_spark

BRONZE = RAIZ / "datos" / "bronze"
SILVER_INAH = RAIZ / "datos" / "silver" / "inah"

# Papel de cada sitio INAH de Quintana Roo en la campaña (docs/regiones/REGIONES.md, D.5).
PAPEL_QROO = {
    "Z.A. de Oxtankah": ("Bahía Calderitas–Oxtankah", "promovida"),
    "Z.A. de Kohunlich": ("Ruta arqueológica del sur", "promovida"),
    "Z.A. de Dzibanché-Kinichná": ("Ruta arqueológica del sur", "promovida"),
    "Z.A de Ichkabal": ("Ruta arqueológica del sur", "promovida"),
    "Z.A. de Tulum": ("Tulum", "referencia"),
    "Z.A. de El Rey": ("Cancún", "referencia"),
    "Z.A. de El Meco": ("Cancún", "referencia"),
    "Museo Maya de Cancún con Z. A": ("Cancún", "referencia"),
    "Z.A. de Xcaret": ("Riviera Maya", "referencia"),
    "Z.A. de Xelhá": ("Riviera Maya", "referencia"),
    "Z.A. de Cobá": ("Cobá + Punta Laguna", "retirada"),
    "Z.A. de Muyil": ("Muyil", "retirada"),
    "Z.A. de Chacchoben": ("Chacchoben", "excluida"),
    "Z.A. de San Gervasio": ("Cozumel", "excluida"),
}
LLAVE = ["estado", "nombre", "anio", "mes", "tipo_visitante"]


def leer_inah() -> pd.DataFrame:
    """Lee el Excel que viene dentro del zip más reciente de Bronze y pone nombres en español y snake_case."""
    zipf = sorted(BRONZE.glob("datatur/*/inah/BdINAH.zip"))[-1]
    with zipfile.ZipFile(zipf) as z:
        nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
        d = pd.read_excel(io.BytesIO(z.read(nombre)))
    d = d.rename(columns={"Clasificación INAH": "clasificacion", "Estado": "estado", "Nombre": "nombre", "Año": "anio",
                          "id_mes": "mes", "Visitantes": "visitantes", "Tipo": "tipo_visitante"})
    d["tipo_visitante"] = d.tipo_visitante.str.lower()
    d["periodo"] = pd.to_datetime(dict(year=d.anio, month=d.mes, day=1)).dt.date
    d["archivo"] = str(zipf.relative_to(RAIZ))
    return d[["clasificacion", "estado", "nombre", "anio", "mes", "periodo", "tipo_visitante", "visitantes", "archivo"]]


def quitar_duplicados(d: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Por cada llave repetida conserva la fila con cifra. Se detiene si hay dos cifras distintas de cero."""
    grupos = d.groupby(LLAVE).visitantes
    conflicto = grupos.apply(lambda v: (v > 0).sum() > 1)
    if conflicto.any():
        raise ValueError(f"INAH: {int(conflicto.sum())} llaves con dos cifras distintas de cero; revisar a mano")
    antes = len(d)
    d = d.sort_values("visitantes", ascending=False).drop_duplicates(LLAVE, keep="first")
    return d.sort_values(LLAVE).reset_index(drop=True), antes - len(d)


def agregar_papel(d: pd.DataFrame) -> pd.DataFrame:
    es_qroo = d.estado == "Quintana Roo"
    d["region_campana"] = d.nombre.map(lambda n: PAPEL_QROO.get(n, (None, None))[0]).where(es_qroo)
    d["papel_campana"] = d.nombre.map(lambda n: PAPEL_QROO.get(n, (None, None))[1]).where(es_qroo)
    d["es_qroo"] = es_qroo
    d["sin_visitantes_flag"] = d.visitantes == 0
    return d


def construir_silver_inah(spark=None):
    spark = spark or crear_spark("silver-inah")
    d = leer_inah()
    d, quitadas = quitar_duplicados(d)
    d = agregar_papel(d)
    df = spark.createDataFrame(d.astype({"region_campana": "object", "papel_campana": "object"}))
    df.write.mode("overwrite").partitionBy("anio").parquet(str(SILVER_INAH))
    q = d[d.es_qroo]
    print(f"INAH Silver: {len(d):,} filas ({quitadas} duplicadas quitadas); Quintana Roo: {len(q):,} filas, "
          f"{q.nombre.nunique()} sitios")
    anual = (q[q.anio == 2025].groupby(["papel_campana", "region_campana"]).visitantes.sum()
             .sort_values(ascending=False))
    print("Visitantes 2025 por región y papel:\n" + anual.to_string())
    return d


In [14]:
%%modulo torre.base.silver_iter
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Limpia el Censo de Población y Vivienda 2020 de Quintana Roo por localidad (INEGI, ITER) y lo
#                    guarda en Silver: población, viviendas habitadas, servicios de las viviendas (agua entubada,
#                    drenaje, electricidad) y coordenadas en grados decimales. A cada localidad le pone la región de la
#                    campaña a la que pertenece (o "referencia"), para calcular turistas por residente.
# Por qué así:       - Decisión de Brandon (28-sep-2026): la población de cada región se mide POR LOCALIDAD, no por
#                      municipio, porque 4 de las 5 regiones están en Othón P. Blanco (233,648 habitantes) y a nivel
#                      municipio no se podrían distinguir. Alternativa descartada: municipio.
#                    - La lista de localidades de cada región es un criterio del proyecto; está en REGION_LOCALIDADES
#                      con su justificación y se defiende en docs/decisiones/05-planteamiento.md.
#                    - Servicios de vivienda = criterio 4 de selección (fragilidad de servicios: agua, drenaje).
#                    - INEGI reserva algunos valores con "*" (confidencialidad) o "N/D": se guardan como nulos con
#                      reservado_flag, nunca como 0 (regla de oro 1).
#                    - Se conservan solo las filas de localidad: las de total estatal (MUN 000), total municipal
#                      (LOC 0000) y "localidades de una o dos viviendas" (LOC 9998/9999) se separan con tipo_fila.
# Datos de entrada:  datos/bronze/iter/<fecha>/iter_23_cpv2020_csv.zip (D7).
# Alimenta a:        Fase 3 (tabla de criterios: turistas por residente y servicios), A1 Radar (presión por residente)
#                    y el mapa de la página (coordenadas de cada región).

import re
import zipfile

import pandas as pd

from torre.base.entorno import RAIZ, crear_spark

BRONZE_ITER = RAIZ / "datos" / "bronze" / "iter"
SILVER_ITER = RAIZ / "datos" / "silver" / "iter"
COLUMNAS = {"MUN": "cve_mun", "NOM_MUN": "municipio", "LOC": "cve_loc", "NOM_LOC": "localidad", "LATITUD": "latitud_gms",
            "LONGITUD": "longitud_gms", "POBTOT": "poblacion", "TVIVPARHAB": "viviendas_habitadas",
            "VPH_AGUADV": "viviendas_con_agua", "VPH_DRENAJ": "viviendas_con_drenaje", "VPH_C_ELEC": "viviendas_con_luz"}
NUMERICAS = ["poblacion", "viviendas_habitadas", "viviendas_con_agua", "viviendas_con_drenaje", "viviendas_con_luz"]

# Región de la campaña → localidades (clave de municipio, clave de localidad, nombre esperado). El nombre se comprueba
# al construir: si INEGI cambia una clave, el proceso se detiene en lugar de asignar la localidad equivocada.
REGION_LOCALIDADES = {
    "Chetumal": [("004", "0001", "Chetumal")],
    "Bahía Calderitas–Oxtankah": [("004", "0016", "Calderitas")],               # Oxtankah está junto a Calderitas
    "Laguna Milagros–Xul-Ha": [("004", "0114", "Xul-Ha"), ("004", "0037", "Huay-Pix")],  # las dos orillas habitadas
    "Ruta arqueológica del sur": [("004", "0064", "Nicolás Bravo"), ("004", "0245", "Morocoy"),
                                  ("004", "0033", "Francisco Villa")],         # poblados de acceso a Kohunlich y Dzibanché
    "Maya Ka'an + Kantemó": [("002", "0001", "Felipe Carrillo Puerto"), ("002", "0250", "Tihosuco"),
                             ("002", "0239", "Señor"), ("002", "0044", "Chunhuhub"), ("006", "0076", "Kantemó")],
    # Referencia (no se promueve): las cabeceras donde hoy está el turista.
    "Cancún (referencia)": [("005", "0001", "Cancún")],
    "Playa del Carmen (referencia)": [("008", "0001", "Playa del Carmen")],
    "Tulum (referencia)": [("009", "0001", "Tulum")],
}


def _grados(texto) -> float | None:
    """Convierte '88°17\\'52.436" W' a grados decimales (−88.2979). Así publica el ITER las coordenadas."""
    if not isinstance(texto, str):
        return None
    m = re.match(r"(\d+)°(\d+)'([\d.]+)\"\s*([NSEW])", texto.strip())
    if not m:
        return None
    g, mi, s, h = m.groups()
    valor = int(g) + int(mi) / 60 + float(s) / 3600
    return -valor if h in "SW" else valor


def leer_iter() -> pd.DataFrame:
    zipf = sorted(BRONZE_ITER.glob("*/iter_23_cpv2020_csv.zip"))[-1]
    with zipfile.ZipFile(zipf) as z:
        nombre = next(n for n in z.namelist() if "conjunto_de_datos" in n and n.endswith(".csv"))
        d = pd.read_csv(z.open(nombre), dtype=str)
    faltan = set(COLUMNAS) - set(d.columns)
    if faltan:
        raise ValueError(f"ITER: no existen las columnas {sorted(faltan)}; revisar el diccionario de datos")
    d = d[list(COLUMNAS)].rename(columns=COLUMNAS)
    d["tipo_fila"] = "localidad"
    d.loc[d.cve_loc == "0000", "tipo_fila"] = "total_municipal"
    d.loc[d.cve_mun == "000", "tipo_fila"] = "total_estatal"
    d.loc[d.cve_loc.isin(["9998", "9999"]), "tipo_fila"] = "localidades_pequenas"
    reservado = d[NUMERICAS].isin(["*", "N/D"]).any(axis=1)
    for c in NUMERICAS:
        d[c] = pd.to_numeric(d[c].where(~d[c].isin(["*", "N/D"])), errors="coerce").astype("Int64")
    d["reservado_flag"] = reservado
    d["latitud"] = d.latitud_gms.map(_grados)
    d["longitud"] = d.longitud_gms.map(_grados)
    d["archivo"] = str(zipf.relative_to(RAIZ))
    return d.drop(columns=["latitud_gms", "longitud_gms"])


def asignar_regiones(d: pd.DataFrame) -> pd.DataFrame:
    d["region_campana"] = None
    d["papel_campana"] = None
    for region, localidades in REGION_LOCALIDADES.items():
        for mun, loc, esperado in localidades:
            fila = (d.cve_mun == mun) & (d.cve_loc == loc)
            if fila.sum() != 1 or d.loc[fila, "localidad"].iloc[0] != esperado:
                encontrado = d.loc[fila, "localidad"].tolist()
                raise ValueError(f"ITER: {mun}-{loc} debía ser {esperado} y es {encontrado}; no se asigna a ciegas")
            d.loc[fila, "region_campana"] = region.replace(" (referencia)", "")
            d.loc[fila, "papel_campana"] = "referencia" if "(referencia)" in region else "promovida"
    return d


def construir_silver_iter(spark=None):
    spark = spark or crear_spark("silver-iter")
    d = asignar_regiones(leer_iter())
    spark.createDataFrame(d.astype({c: "float64" for c in NUMERICAS}).astype(
        {"region_campana": "object", "papel_campana": "object"})).write.mode("overwrite").parquet(str(SILVER_ITER))
    loc = d[d.tipo_fila == "localidad"]
    estatal = int(d.loc[d.tipo_fila == "total_estatal", "poblacion"].iloc[0])
    print(f"ITER Silver: {len(d):,} filas; {len(loc):,} localidades; población estatal {estatal:,}; "
          f"{int(loc.reservado_flag.sum()):,} localidades con algún dato reservado por INEGI")
    resumen = (d[d.region_campana.notna()].groupby(["papel_campana", "region_campana"])
               [["poblacion", "viviendas_habitadas", "viviendas_con_agua", "viviendas_con_drenaje"]].sum())
    print(resumen.to_string())
    return d


In [15]:
%%modulo torre.base.silver_clima
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Limpia el clima de Open-Meteo (reanálisis ERA5) de 8 puntos de Quintana Roo y lo guarda en Silver en
#                    dos tablas: clima_diario (1950–2026: temperatura máxima, lluvia del día y viento máximo) y
#                    clima_horario (2019–2026: temperatura, lluvia y viento de cada hora).
# Por qué así:       - Dos tablas porque responden preguntas distintas: la diaria da 76 años para medir la temporada de
#                      lluvias (variable del Pronóstico mensual) y la horaria alimenta el flujo casi en tiempo real de
#                      la Torre en vivo (Fase 7). Alternativa descartada: una sola tabla con huecos donde no hay hora.
#                    - Horas: el crudo horario viene en UTC (utc_offset_seconds = 0) y el diario ya viene en hora de
#                      Cancún. Se guardan las dos horas en la tabla horaria. Quintana Roo usa UTC−5 fijo desde el
#                      1-feb-2015, así que en 2019–2026 la hora local es UTC − 5 h sin cambios de horario.
#                    - Revisión del 30-sep-2026: 128 archivos, 224,232 filas diarias y 542,784 horarias, 0 nulos. Si
#                      algún día llega un nulo, se conserva nulo con sin_dato_flag (regla de oro 1: no se rellena).
#                    - Es reanálisis (modelo meteorológico que asimila observaciones), no una estación: se declara en
#                      la columna fuente_tipo. No lleva _est porque es un producto oficial, no un cálculo nuestro.
#                    - Cada punto lleva su papel en la campaña (regla de oro 9): promovida, referencia o excluida.
#                    - JSON → pandas → Spark escribe el Parquet particionado por año.
# Datos de entrada:  datos/bronze/clima/<fecha>/diario/<punto>_<década>s.json y horario/<punto>_<año>.json (D8).
# Alimenta a:        A3 Pronóstico (lluvia y temperatura como variables del modelo con clima; en qué meses la lluvia
#                    cambia el mensaje de la campaña) y A5 Torre en vivo (alertas de lluvia o viento de la semana).

import json

import pandas as pd

from torre.base.entorno import RAIZ, crear_spark

BRONZE = RAIZ / "datos" / "bronze"
SILVER_DIARIO = RAIZ / "datos" / "silver" / "clima_diario"
SILVER_HORARIO = RAIZ / "datos" / "silver" / "clima_horario"

# Papel de cada punto (docs/regiones/REGIONES.md; regla de oro 9 de CLAUDE.md).
PAPEL_PUNTO = {
    "chetumal": ("Chetumal", "promovida"),
    "kohunlich": ("Ruta arqueológica del sur", "promovida"),
    "felipe_carrillo_puerto": ("Maya Ka'an + Kantemó", "promovida"),
    "cancun": ("Cancún", "referencia"),
    "playa_del_carmen": ("Riviera Maya", "referencia"),
    "tulum": ("Tulum", "referencia"),
    "bacalar": ("Bacalar", "excluida"),
    "coba": ("Cobá + Punta Laguna", "excluida"),
}
NOMBRES = {"temperature_2m_max": "temp_max_c", "precipitation_sum": "lluvia_mm", "wind_speed_10m_max": "viento_max_kmh",
           "temperature_2m": "temp_c", "precipitation": "lluvia_mm", "wind_speed_10m": "viento_kmh"}


def _leer(tipo: str) -> pd.DataFrame:
    """Une todos los archivos de un tipo ('diario' u 'horario') de la descarga más reciente."""
    carpeta = sorted(BRONZE.glob("clima/*/"))[-1] / tipo
    partes = []
    for archivo in sorted(carpeta.glob("*.json")):
        crudo = json.loads(archivo.read_text(encoding="utf-8"))
        clave = "daily" if tipo == "diario" else "hourly"
        p = pd.DataFrame(crudo[clave]).rename(columns=NOMBRES)
        p["punto"] = archivo.stem.rsplit("_", 1)[0]
        p["lat"], p["lon"] = crudo["latitude"], crudo["longitude"]  # celda de la malla ERA5 que devolvió la API
        p["archivo"] = str(archivo.relative_to(RAIZ))
        partes.append(p)
    d = pd.concat(partes, ignore_index=True)
    duplicadas = d.duplicated(["punto", "time"]).sum()
    if duplicadas:  # regla de oro 5: dos lecturas del mismo punto y hora no se resuelven solas
        raise ValueError(f"Clima {tipo}: {duplicadas} filas repetidas por punto y tiempo")
    return d


def _papel(d: pd.DataFrame) -> pd.DataFrame:
    d["region_campana"] = d.punto.map(lambda p: PAPEL_PUNTO[p][0])
    d["papel_campana"] = d.punto.map(lambda p: PAPEL_PUNTO[p][1])
    d["fuente_tipo"] = "reanálisis ERA5 (Open-Meteo)"
    return d


def clima_diario() -> pd.DataFrame:
    d = _leer("diario")
    d["fecha"] = pd.to_datetime(d.pop("time")).dt.date
    d["anio"] = pd.to_datetime(d.fecha).dt.year
    d["mes"] = pd.to_datetime(d.fecha).dt.month
    d["sin_dato_flag"] = d[["temp_max_c", "lluvia_mm", "viento_max_kmh"]].isna().any(axis=1)
    return _papel(d)[["punto", "region_campana", "papel_campana", "fecha", "anio", "mes", "temp_max_c", "lluvia_mm",
                      "viento_max_kmh", "sin_dato_flag", "lat", "lon", "fuente_tipo", "archivo"]]


def clima_horario() -> pd.DataFrame:
    d = _leer("horario")
    d["fecha_hora_utc"] = pd.to_datetime(d.pop("time"))
    d["fecha_hora_local"] = d.fecha_hora_utc - pd.Timedelta(hours=5)  # UTC−5 fijo en Q. Roo desde 2015
    d["anio"] = d.fecha_hora_local.dt.year
    d["sin_dato_flag"] = d[["temp_c", "lluvia_mm", "viento_kmh"]].isna().any(axis=1)
    return _papel(d)[["punto", "region_campana", "papel_campana", "fecha_hora_utc", "fecha_hora_local", "anio", "temp_c",
                      "lluvia_mm", "viento_kmh", "sin_dato_flag", "lat", "lon", "fuente_tipo", "archivo"]]


def construir_silver_clima(spark=None):
    spark = spark or crear_spark("silver-clima")
    dia, hora = clima_diario(), clima_horario()
    spark.createDataFrame(dia).write.mode("overwrite").partitionBy("anio").parquet(str(SILVER_DIARIO))
    spark.createDataFrame(hora).write.mode("overwrite").partitionBy("anio").parquet(str(SILVER_HORARIO))
    print(f"Clima Silver: diario {len(dia):,} filas ({dia.fecha.min()} → {dia.fecha.max()}), "
          f"horario {len(hora):,} filas; puntos: {dia.punto.nunique()}; sin dato: {int(dia.sin_dato_flag.sum())} días, "
          f"{int(hora.sin_dato_flag.sum())} horas")
    lluvia = (dia[(dia.punto == "chetumal") & dia.anio.between(1991, 2020)]
              .groupby(["anio", "mes"]).lluvia_mm.sum().groupby("mes").mean().round(0))
    print("Chetumal, lluvia media mensual 1991–2020 (mm):", lluvia.to_dict())
    return dia, hora


In [16]:
%%modulo torre.base.silver_huracanes
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Limpia HURDAT2 (NOAA, todas las tormentas del Atlántico 1851–2025) y la guarda en Silver con una fila
#                    por punto de trayectoria (cada 6 horas). A cada punto le calcula su distancia a Chetumal y le pone
#                    las banderas que definen "una tormenta que afecta al sur de Quintana Roo".
# Por qué así:       - Definición elegida por Brandon (30-sep-2026, docs/decisiones/10-silver-fase5.md): un evento es una
#                      tormenta con al menos un punto a ≤ 200 km de Chetumal y viento ≥ 34 nudos (tormenta tropical o
#                      huracán), contada desde 1966 (inicio de la vigilancia por satélite): 31 eventos en 60 años.
#                      Alternativas descartadas: 100 km (13 eventos desde 1966: muy pocos para una tasa por mes),
#                      300 km (59 eventos: incluye tormentas que pasan por el norte de Yucatán y por Honduras) y contar
#                      desde 1851 (0.469 por año contra 0.517: antes de los satélites se perdían tormentas en el mar).
#                    - Se guardan TODOS los puntos (no solo los del sur) para que el radio o el umbral se puedan
#                      cambiar sin volver a leer el texto crudo (análisis de sensibilidad en la Fase 5).
#                    - Parser con expresión regular sobre la posición, el viento y la presión: 2 de 55,524 líneas
#                      vienen mal formadas en el archivo oficial. 1969-09-29 06Z trae "63.3N    7.5E" sin coma (se lee
#                      completa) y 1975-12-07 00Z trae "38.83" sin hemisferio (la latitud queda nula; no se adivina).
#                      Ambas se marcan con formato_irregular_flag. Las dos están en el Atlántico norte, a más de
#                      3,000 km de Chetumal: no cambian el conteo.
#                    - Faltantes oficiales de HURDAT2: viento -99 y presión -999 se guardan como nulos, no como números.
#                    - Texto → pandas (el formato alterna encabezados y filas) y Spark escribe el Parquet por año.
# Datos de entrada:  datos/bronze/huracanes/<fecha>/hurdat2-1851-2025-*.txt (D9).
# Alimenta a:        A3 Pronóstico: probabilidad mensual de tormenta (Poisson) y choques del Monte Carlo, que deciden en
#                    qué meses la campaña reserva presupuesto de contingencia.

import math
import re

import pandas as pd

from torre.base.entorno import RAIZ, crear_spark

BRONZE = RAIZ / "datos" / "bronze"
SILVER_HURACANES = RAIZ / "datos" / "silver" / "huracanes"

CHETUMAL = (18.50, -88.30)     # centro del sur de Quintana Roo (bahía de Chetumal)
RADIO_KM = 200                 # decisión de Brandon, 30-sep-2026
VIENTO_MIN_KT = 34             # 34 nudos ≈ 63 km/h: umbral oficial de tormenta tropical
ANIO_SATELITAL = 1966          # primer año con vigilancia por satélite continua

# Posición + viento + presión. Tolera "18.5N, 88.3W" (normal) y "63.3N    7.5E" (sin coma entre las dos).
POS = re.compile(r"(\d+\.\d)([NS])[\s,]*(\d+\.\d)([EW])\s*,\s*(-?\d+)\s*,\s*(-?\d+)")
# Latitud sin hemisferio ("38.83,  51.0W", tormenta de 1975): la latitud no se adivina, queda nula.
POS_SIN_HEMISFERIO = re.compile(r"(\d+\.\d+)\s*,\s*(\d+\.\d)([EW])\s*,\s*(-?\d+)\s*,\s*(-?\d+)")
LAT_BIEN = re.compile(r"\d+\.\d[NS]")


def km_haversine(lat1, lon1, lat2, lon2) -> float:
    """Distancia sobre la esfera terrestre (radio 6,371 km). Ecuación en ECUACIONES.md §3.1."""
    p = math.pi / 180
    a = (math.sin((lat2 - lat1) * p / 2) ** 2
         + math.cos(lat1 * p) * math.cos(lat2 * p) * math.sin((lon2 - lon1) * p / 2) ** 2)
    return 2 * 6371 * math.asin(math.sqrt(a))


def leer_hurdat2() -> pd.DataFrame:
    """Recorre el archivo: una línea de encabezado (id, nombre, n puntos) seguida de n líneas de trayectoria."""
    archivo = sorted(BRONZE.glob("huracanes/*/hurdat2-1851-*.txt"))[-1]
    lineas = archivo.read_text(encoding="utf-8").splitlines()
    filas, i = [], 0
    while i < len(lineas):
        cab = [x.strip() for x in lineas[i].split(",")]
        id_tormenta, nombre, n = cab[0], cab[1], int(cab[2])
        for linea in lineas[i + 1:i + 1 + n]:
            c = [x.strip() for x in linea.split(",")]
            m = POS.search(linea)
            if m:
                lat = float(m.group(1)) * (1 if m.group(2) == "N" else -1)
                lon_txt, hemi_lon, viento, presion = m.group(3), m.group(4), int(m.group(5)), int(m.group(6))
            else:
                m = POS_SIN_HEMISFERIO.search(linea)
                if not m:  # regla de oro 5: una línea que no se entiende detiene el proceso
                    raise ValueError(f"HURDAT2: línea sin posición legible en {id_tormenta}: {linea[:60]}")
                lat = None
                lon_txt, hemi_lon, viento, presion = m.group(2), m.group(3), int(m.group(4)), int(m.group(5))
            filas.append({
                "id_tormenta": id_tormenta, "nombre": nombre,
                "fecha": c[0], "hora_utc": c[1], "marca": c[2] or None, "estado_sistema": c[3],
                "lat": lat, "lon": float(lon_txt) * (-1 if hemi_lon == "W" else 1),
                "viento_kt": None if viento == -99 else viento,
                "presion_mb": None if presion == -999 else presion,
                "formato_irregular_flag": not LAT_BIEN.fullmatch(c[4]),
            })
        i += 1 + n
    d = pd.DataFrame(filas)
    d["anio"] = d.fecha.str[:4].astype(int)
    d["mes"] = d.fecha.str[4:6].astype(int)
    d["fecha_hora_utc"] = pd.to_datetime(d.fecha + d.hora_utc.str.zfill(4), format="%Y%m%d%H%M")
    # Spark en Windows no convierte marcas de tiempo anteriores a 1970 (mktime): se guarda fecha + hora en número.
    d["fecha"] = d.fecha_hora_utc.dt.date
    d["hora_utc"] = d.hora_utc.astype(int)
    d["archivo"] = str(archivo.relative_to(RAIZ))
    return d


def agregar_banderas(d: pd.DataFrame) -> pd.DataFrame:
    d["km_chetumal"] = [None if pd.isna(a) else round(km_haversine(a, b, *CHETUMAL), 1) for a, b in zip(d.lat, d.lon)]
    d["dentro_radio_flag"] = d.km_chetumal.fillna(float("inf")) <= RADIO_KM  # sin latitud no se puede afirmar cercanía
    d["tormenta_o_huracan_flag"] = d.viento_kt.fillna(0) >= VIENTO_MIN_KT
    d["era_satelital_flag"] = d.anio >= ANIO_SATELITAL
    d["afecta_sur_flag"] = d.dentro_radio_flag & d.tormenta_o_huracan_flag & d.era_satelital_flag
    return d


def eventos_sur(d: pd.DataFrame) -> pd.DataFrame:
    """Una fila por tormenta que afecta al sur: mes del primer punto que cumple la definición, viento máximo dentro
    del radio y distancia mínima a Chetumal. Es la serie que usa el modelo de Poisson de la Fase 5."""
    x = d[d.afecta_sur_flag].sort_values(["fecha", "hora_utc"])
    return (x.groupby("id_tormenta")
             .agg(nombre=("nombre", "first"), anio=("anio", "first"), mes=("mes", "first"),
                  viento_max_kt=("viento_kt", "max"), km_min=("km_chetumal", "min"))
             .reset_index().sort_values(["anio", "mes"]).reset_index(drop=True))


def construir_silver_huracanes(spark=None):
    spark = spark or crear_spark("silver-huracanes")
    d = agregar_banderas(leer_hurdat2())
    columnas = ["id_tormenta", "nombre", "anio", "mes", "fecha", "hora_utc", "marca", "estado_sistema", "lat", "lon",
                "viento_kt", "presion_mb", "km_chetumal", "dentro_radio_flag", "tormenta_o_huracan_flag",
                "era_satelital_flag", "afecta_sur_flag", "formato_irregular_flag", "archivo"]
    d = d[columnas]
    df = spark.createDataFrame(d.astype({"viento_kt": "Int64", "presion_mb": "Int64", "marca": "object"}))
    df.write.mode("overwrite").partitionBy("anio").parquet(str(SILVER_HURACANES))
    ev = eventos_sur(d)
    anios = d.anio.max() - ANIO_SATELITAL + 1
    print(f"HURDAT2 Silver: {len(d):,} puntos de {d.id_tormenta.nunique():,} tormentas ({d.anio.min()}–{d.anio.max()}); "
          f"{int(d.formato_irregular_flag.sum())} líneas con formato irregular")
    print(f"Afectan al sur (≤{RADIO_KM} km de Chetumal, ≥{VIENTO_MIN_KT} kt, desde {ANIO_SATELITAL}): "
          f"{len(ev)} eventos en {anios} años = {len(ev) / anios:.3f} por año")
    print("Eventos por mes:", ev.mes.value_counts().sort_index().to_dict())
    return d


In [17]:
%%modulo torre.base.silver_fred
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Limpia dos series de FRED (Reserva Federal de St. Louis) y las guarda en Silver:
#                    fred_diario  → tipo de cambio pesos por dólar (DEXMXUS), un dato por día hábil desde 1993;
#                    fred_mensual → promedio mensual del tipo de cambio y el índice de precios al consumidor de EE. UU.
#                    (CPIAUCSL), una fila por mes.
# Por qué así:       - Los días sin cotización (fines de semana no vienen; feriados de EE. UU. vienen vacíos: 336 de
#                      8,575 filas) se conservan vacíos con sin_dato_flag. No se copia el día anterior ni se interpola
#                      (regla de oro 1).
#                    - El promedio mensual usa solo los días observados y guarda cuántos fueron (n_dias_observados),
#                      para que se vea en qué se apoya cada cifra. Alternativa descartada: último dato del mes (depende
#                      de un solo día).
#                    - El mes en curso de 2026 se marca con mes_incompleto_flag: su promedio todavía puede cambiar.
#                    - CSV → pandas → Spark escribe el Parquet.
# Datos de entrada:  datos/bronze/fred/<fecha>/DEXMXUS.csv y CPIAUCSL.csv (D10).
# Alimenta a:        A3 Pronóstico: sensibilidad del escenario al tipo de cambio ("¿qué pasa si cambia el dólar?", que
#                    mueve la llegada de visitantes de EE. UU.) y deflactar costos en dólares de los canales (D13).

import pandas as pd

from torre.base.entorno import RAIZ, crear_spark

BRONZE = RAIZ / "datos" / "bronze"
SILVER_DIARIO = RAIZ / "datos" / "silver" / "fred_diario"
SILVER_MENSUAL = RAIZ / "datos" / "silver" / "fred_mensual"


def _serie(nombre: str) -> pd.DataFrame:
    archivo = sorted(BRONZE.glob(f"fred/*/{nombre}.csv"))[-1]
    d = pd.read_csv(archivo).rename(columns={"observation_date": "fecha", nombre: "valor"})
    d["fecha"] = pd.to_datetime(d.fecha)
    d["valor"] = pd.to_numeric(d.valor, errors="coerce")  # celda vacía → nulo (no cero)
    if d.fecha.duplicated().any():
        raise ValueError(f"FRED {nombre}: fechas repetidas")
    d["archivo"] = str(archivo.relative_to(RAIZ))
    return d


def tipo_cambio_diario() -> pd.DataFrame:
    d = _serie("DEXMXUS").rename(columns={"valor": "pesos_por_dolar"})
    d["anio"], d["mes"] = d.fecha.dt.year, d.fecha.dt.month
    d["sin_dato_flag"] = d.pesos_por_dolar.isna()
    d["fecha"] = d.fecha.dt.date
    return d[["fecha", "anio", "mes", "pesos_por_dolar", "sin_dato_flag", "archivo"]]


def mensual(diario: pd.DataFrame) -> pd.DataFrame:
    tc = (diario.groupby(["anio", "mes"])
          .agg(pesos_por_dolar=("pesos_por_dolar", "mean"),       # mean ignora los nulos: solo días observados
               n_dias_observados=("pesos_por_dolar", "count"),
               n_dias_sin_dato=("sin_dato_flag", "sum"))
          .reset_index())
    ultimo = pd.Timestamp(max(diario.fecha))
    tc["mes_incompleto_flag"] = (tc.anio == ultimo.year) & (tc.mes == ultimo.month)
    cpi = _serie("CPIAUCSL").rename(columns={"valor": "inflacion_eeuu_indice"})
    cpi["anio"], cpi["mes"] = cpi.fecha.dt.year, cpi.fecha.dt.month
    m = cpi[["anio", "mes", "inflacion_eeuu_indice"]].merge(tc, on=["anio", "mes"], how="outer")
    m["periodo"] = pd.to_datetime(dict(year=m.anio, month=m.mes, day=1)).dt.date
    m["mes_incompleto_flag"] = m.mes_incompleto_flag.fillna(False).astype(bool)
    return m.sort_values("periodo").reset_index(drop=True)[
        ["periodo", "anio", "mes", "pesos_por_dolar", "n_dias_observados", "n_dias_sin_dato", "mes_incompleto_flag",
         "inflacion_eeuu_indice"]]


def construir_silver_fred(spark=None):
    spark = spark or crear_spark("silver-fred")
    dia = tipo_cambio_diario()
    mes = mensual(dia)
    spark.createDataFrame(dia).write.mode("overwrite").partitionBy("anio").parquet(str(SILVER_DIARIO))
    spark.createDataFrame(mes.astype({"n_dias_observados": "Int64", "n_dias_sin_dato": "Int64"})) \
        .write.mode("overwrite").parquet(str(SILVER_MENSUAL))
    print(f"FRED Silver: {len(dia):,} días de tipo de cambio ({dia.fecha.min()} → {dia.fecha.max()}), "
          f"{int(dia.sin_dato_flag.sum())} sin dato; {len(mes):,} meses")
    print(mes[mes.anio >= 2025].tail(6).to_string(index=False))
    return dia, mes


In [18]:
%%modulo torre.base.silver_nacionalidad
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Limpia la base de llegadas de extranjeros por avión según su nacionalidad (DataTur, BD_Nacionalidad,
#                    con cifras de la Unidad de Política Migratoria) y la guarda en Silver con una fila por mes,
#                    aeropuerto, país y sexo. A los aeropuertos de Quintana Roo les pone su lugar y su papel en la campaña.
# Por qué así:       - Es la única fuente que dice DE DÓNDE viene el visitante extranjero en cada aeropuerto del estado:
#                      la base del buyer persona (Fase 8) y de los idiomas de la página ("Mercados que llegan").
#                    - Solo cuenta extranjeros: no hay país "México" en la base (verificado el 02-oct-2026). Por eso la
#                      columna se llama llegadas_extranjeros y no "pasajeros".
#                    - Revisado el 02-oct-2026: 0 llaves repetidas y 0 filas idénticas (521,363 filas, 2012 → jul-2026).
#                      Si un día aparecen, el proceso se detiene (regla de oro 5).
#                    - El aeropuerto de Tulum abre en feb-2024: antes no hay filas, y eso no es un cero.
#                    - Alternativa descartada: guardar solo Quintana Roo. Se conserva el país para comparar.
# Datos de entrada:  datos/bronze/datatur/<fecha>/nacionalidad/BD_Nacionalidad.zip (D3).
# Alimenta a:        Fase 8 (buyer persona: país de origen por aeropuerto y temporada) y la Fase 6 (en qué mercado se
#                    anuncia).

import io
import zipfile

import pandas as pd

from torre.base.entorno import RAIZ, crear_spark

BRONZE = RAIZ / "datos" / "bronze"
SILVER_NACIONALIDAD = RAIZ / "datos" / "silver" / "nacionalidad"

# Aeropuertos de Quintana Roo → (lugar, papel en la campaña), igual que en docs/regiones/REGIONES.md.
AEROPUERTOS_QROO = {
    "Chetumal, Q. Roo": ("Chetumal", "promovida"),
    "Cancún, Q. Roo": ("Cancún", "referencia"),
    "A.I Tulum Felipe Carrillo Puerto, Q. Roo.": ("Tulum", "referencia"),
    "Cozumel, Q. Roo": ("Cozumel", "excluida"),
}
LLAVE = ["periodo", "aeropuerto", "pais", "sexo"]


def leer_nacionalidad() -> pd.DataFrame:
    """Lee el Excel del zip más reciente y pone nombres en español y snake_case."""
    zipf = sorted(BRONZE.glob("datatur/*/nacionalidad/BD_Nacionalidad.zip"))[-1]
    with zipfile.ZipFile(zipf) as z:
        nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
        d = pd.read_excel(io.BytesIO(z.read(nombre)))
    d = d.rename(columns={"Año": "anio", "MesNum": "mes", "Aeropuerto": "aeropuerto", "Pais": "pais",
                          "Región": "region_mundo", "Sexo": "sexo", "Valor": "llegadas_extranjeros"})
    d["periodo"] = d.Fecha.dt.date
    d["sexo"] = d.sexo.str.lower()
    d["archivo"] = str(zipf.relative_to(RAIZ))
    return d[["anio", "mes", "periodo", "aeropuerto", "pais", "region_mundo", "sexo", "llegadas_extranjeros", "archivo"]]


def revisar_llave(d: pd.DataFrame) -> None:
    """Regla de oro 5: si hay dos filas con la misma llave, no se elige una; se detiene."""
    repetidas = int(d.duplicated(LLAVE).sum())
    if repetidas:
        raise ValueError(f"Nacionalidad: {repetidas} llaves repetidas; revisar a mano antes de seguir")


def agregar_papel(d: pd.DataFrame) -> pd.DataFrame:
    d["es_qroo"] = d.aeropuerto.isin(AEROPUERTOS_QROO)
    d["lugar_campana"] = d.aeropuerto.map(lambda a: AEROPUERTOS_QROO.get(a, (None, None))[0])
    d["papel_campana"] = d.aeropuerto.map(lambda a: AEROPUERTOS_QROO.get(a, (None, None))[1])
    return d


def construir_silver_nacionalidad(spark=None) -> pd.DataFrame:
    spark = spark or crear_spark("silver-nacionalidad")
    d = leer_nacionalidad()
    revisar_llave(d)
    d = agregar_papel(d)
    df = spark.createDataFrame(d.astype({"lugar_campana": "object", "papel_campana": "object"}))
    df.write.mode("overwrite").partitionBy("anio").parquet(str(SILVER_NACIONALIDAD))
    q = d[d.es_qroo]
    print(f"Nacionalidad Silver: {len(d):,} filas; Quintana Roo: {len(q):,} filas")
    print("Llegadas de extranjeros por año (Quintana Roo):\n"
          + q.pivot_table(index="anio", columns="lugar_campana", values="llegadas_extranjeros", aggfunc="sum")
          .fillna(0).astype(int).to_string())
    return d


In [19]:
%%modulo torre.base.silver_afac
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Limpia la base de pasajeros por aerolínea de la AFAC (DataTur, DB_AFAC) y la guarda en Silver con
#                    una fila por mes, tipo de vuelo, servicio, región de la aerolínea y aerolínea.
# Por qué así:       - Es la serie nacional de pasajeros aéreos 2016 → jul-2026: sirve de contexto para saber si el turismo
#                      aéreo del país crece o cae (2020: 48.4 millones; 2025: 122.7 millones).
#                    - Duplicados revisados el 02-oct-2026 (300 llaves repetidas):
#                        · 98 filas idénticas → se deja una.
#                        · 177 llaves con una cifra y una copia en cero → se conserva la cifra (misma regla que INAH).
#                        · 25 llaves con DOS cifras: todas de "Virgin América (Alaska Airlines)", de 2016 a ene-2018. Son
#                          dos aerolíneas que se fusionaron en 2018 bajo una sola etiqueta. Decisión de Brandon: se SUMAN
#                          y la fila lleva sumada_flag. En juego hay 369,311 pasajeros de 1,042 millones (0.035 %).
#                        · Si aparece otra llave con dos cifras, el proceso se detiene (regla de oro 5).
#                    - El texto "Fletamento " trae un espacio de más; se recorta.
#                    - 46 % de las filas valen 0: son aerolíneas del catálogo sin vuelos ese mes. Se conservan.
# Datos de entrada:  datos/bronze/datatur/<fecha>/afac/DB_AFAC.zip (D4).
# Alimenta a:        Fase 8 (contexto de mercados: qué región de aerolíneas crece) y el informe (grandes volúmenes).

import io
import zipfile

import pandas as pd

from torre.base.entorno import RAIZ, crear_spark

BRONZE = RAIZ / "datos" / "bronze"
SILVER_AFAC = RAIZ / "datos" / "silver" / "afac"

LLAVE = ["anio", "mes", "tipo_vuelo", "servicio", "region_aerolinea", "aerolinea"]
# Única etiqueta donde dos cifras en el mismo mes se suman (decisión de Brandon del 02-oct-2026).
SE_SUMAN = {"Virgin América (Alaska Airlines)"}


def leer_afac() -> pd.DataFrame:
    zipf = sorted(BRONZE.glob("datatur/*/afac/DB_AFAC.zip"))[-1]
    with zipfile.ZipFile(zipf) as z:
        nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
        d = pd.read_excel(io.BytesIO(z.read(nombre)))
    d = d.rename(columns={"Año": "anio", "Id_mes": "mes", "Tipo": "tipo_vuelo", "Servicio": "servicio",
                          "Región": "region_aerolinea", "Aerolinea": "aerolinea", "Pasajeros": "pasajeros"})
    for c in ["tipo_vuelo", "servicio", "region_aerolinea", "aerolinea"]:
        d[c] = d[c].str.strip()
    d["tipo_vuelo"] = d.tipo_vuelo.str.lower()
    d["servicio"] = d.servicio.str.lower()
    d["periodo"] = d.Fecha.dt.date
    d["archivo"] = str(zipf.relative_to(RAIZ))
    return d[LLAVE + ["periodo", "pasajeros", "archivo"]]


def quitar_duplicados(d: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Aplica las tres reglas de duplicados. Devuelve la tabla limpia y cuántas filas tocó cada regla."""
    antes = len(d)
    d = d.drop_duplicates()
    identicas = antes - len(d)
    con_cifra = d.groupby(LLAVE).pasajeros.transform(lambda v: (v > 0).sum())
    dos = d[con_cifra > 1]
    otras = set(dos.aerolinea) - SE_SUMAN
    if otras:
        raise ValueError(f"AFAC: dos cifras en la misma llave para {sorted(otras)}; revisar a mano")
    # Se quitan las copias en cero cuando existe la cifra, y se suman las dos cifras de las etiquetas permitidas.
    d = d[~((con_cifra >= 1) & (d.pasajeros == 0))]
    n_llaves = d.groupby(LLAVE).size()
    sumadas = n_llaves[n_llaves > 1]
    d = (d.groupby(LLAVE + ["periodo", "archivo"], as_index=False, observed=True).pasajeros.sum())
    d["sumada_flag"] = d.set_index(LLAVE).index.isin(sumadas.index)
    # Llaves que solo tenían copias en cero: quedan con una sola fila en 0.
    resumen = {"identicas": identicas, "llaves_sumadas": int(len(sumadas)), "filas_finales": len(d)}
    return d.sort_values(LLAVE).reset_index(drop=True), resumen


def construir_silver_afac(spark=None) -> pd.DataFrame:
    spark = spark or crear_spark("silver-afac")
    d, r = quitar_duplicados(leer_afac())
    spark.createDataFrame(d).write.mode("overwrite").partitionBy("anio").parquet(str(SILVER_AFAC))
    print(f"AFAC Silver: {r['filas_finales']:,} filas; {r['identicas']} idénticas quitadas; "
          f"{r['llaves_sumadas']} llaves sumadas (Virgin América + Alaska)")
    print("Pasajeros por año:\n" + d.groupby("anio").pasajeros.sum().to_string())
    return d


In [20]:
%%modulo torre.base.silver_cruceros
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Limpia la base de cruceros por puerto (DataTur, BaseDatosCruceros: arribos y pasajeros por mes,
#                    2016 → jul-2026) y la guarda en Silver. Después la compara con los cruceristas de SITUR-Q (la otra
#                    fuente del mismo dato) y guarda la diferencia en Gold.
# Por qué así:       - Los cruceros son la mayor presión de llegada del sur del estado (Mahahual: 2.38 millones de
#                      pasajeros en 2025) y alimentan la variable llegadas_x_cuarto del Radar.
#                    - Revisado el 02-oct-2026: 0 llaves repetidas. Cancún, Playa del Carmen, Puerto Morelos y Punta
#                      Venado vienen con 0 en TODOS los meses: están en el catálogo pero no reciben cruceros. Se conservan
#                      con puerto_sin_cruceros_flag; no se borran ni se rellenan.
#                    - Reconciliación (incidente de Big Data, "fuentes que no coinciden"): se compara mes a mes con SITUR-Q
#                      y se declara la diferencia. No se elige una fuente nueva: el Radar sigue con SITUR-Q (Fase 4).
# Datos de entrada:  datos/bronze/datatur/<fecha>/cruceros/BaseDatosCruceros.zip (D4); datos/silver/siturq (D1).
# Alimenta a:        A1 Radar (presión de llegada por cruceros), Fase 7 (vigilancia semanal de puertos) y el informe
#                    (reconciliación entre fuentes).

import io
import zipfile

import pandas as pd

from torre.base.entorno import RAIZ, crear_spark

BRONZE = RAIZ / "datos" / "bronze"
SILVER = RAIZ / "datos" / "silver"
SILVER_CRUCEROS = SILVER / "cruceros"
GOLD_RECONCILIACION = RAIZ / "datos" / "gold" / "reconciliacion_cruceros.parquet"

LLAVE = ["puerto", "anio", "mes"]
# Puertos de Quintana Roo → papel en la campaña (docs/regiones/REGIONES.md: Mahahual y Cozumel, excluidos).
PAPEL_QROO = {"Mahahual": "excluida", "Cozumel": "excluida", "Cancún": "referencia", "Playa del Carmen": "referencia",
              "Puerto Morelos": "excluida", "Punta Venado": "referencia"}
# Nombre del mismo puerto en SITUR-Q (columna unidad).
EN_SITURQ = {"Cozumel": "Cozumel", "Mahahual": "Mahahual"}


def leer_cruceros() -> pd.DataFrame:
    zipf = sorted(BRONZE.glob("datatur/*/cruceros/BaseDatosCruceros.zip"))[-1]
    with zipfile.ZipFile(zipf) as z:
        nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
        d = pd.read_excel(io.BytesIO(z.read(nombre)))
    d = d.rename(columns={"Estado": "estado", "Puerto": "puerto", "Año": "anio", "Id_mes": "mes", "Arribos": "arribos",
                          "Pasajeros": "pasajeros", "Latitud": "lat", "Longitud": "lon", "Región": "litoral"})
    d["periodo"] = d.Fecha.dt.date
    d["archivo"] = str(zipf.relative_to(RAIZ))
    if d.duplicated(LLAVE).any():
        raise ValueError("Cruceros: llaves repetidas; revisar a mano")
    return d[["estado", "puerto", "anio", "mes", "periodo", "arribos", "pasajeros", "lat", "lon", "litoral", "archivo"]]


def agregar_banderas(d: pd.DataFrame) -> pd.DataFrame:
    total = d.groupby("puerto").pasajeros.transform("sum")
    d["puerto_sin_cruceros_flag"] = total == 0
    d["es_qroo"] = d.estado == "Quintana Roo"
    d["papel_campana"] = d.puerto.map(PAPEL_QROO).where(d.es_qroo)
    return d


def reconciliar_siturq(d: pd.DataFrame) -> pd.DataFrame:
    """Compara pasajeros de DataTur con cruceristas de SITUR-Q, mes a mes, en los puertos que tienen ambas fuentes."""
    s = pd.read_parquet(SILVER / "siturq")
    s = s[(s.indicador == "cruceristas") & ~s.hueco_flag]
    s = s.groupby(["unidad", "periodo"], observed=True).valor.sum().reset_index()
    filas = []
    for puerto, unidad in EN_SITURQ.items():
        dt = d[d.puerto == puerto][["periodo", "pasajeros"]]
        st = s[s.unidad == unidad][["periodo", "valor"]].rename(columns={"valor": "cruceristas_siturq"})
        st["periodo"] = pd.to_datetime(st.periodo).dt.date
        m = dt.merge(st, on="periodo", how="inner")
        m.insert(0, "puerto", puerto)
        filas.append(m.rename(columns={"pasajeros": "pasajeros_datatur"}))
    r = pd.concat(filas, ignore_index=True)
    r["diferencia"] = r.cruceristas_siturq - r.pasajeros_datatur
    r["diferencia_pct"] = (r.diferencia / r.pasajeros_datatur.where(r.pasajeros_datatur > 0) * 100).round(2)
    return r


def construir_silver_cruceros(spark=None) -> pd.DataFrame:
    spark = spark or crear_spark("silver-cruceros")
    d = agregar_banderas(leer_cruceros())
    spark.createDataFrame(d.astype({"papel_campana": "object"})).write.mode("overwrite").partitionBy("anio") \
        .parquet(str(SILVER_CRUCEROS))
    q = d[d.es_qroo]
    print(f"Cruceros Silver: {len(d):,} filas; Quintana Roo: {len(q):,} filas, "
          f"{q[q.puerto_sin_cruceros_flag].puerto.nunique()} puertos sin cruceros en toda la serie")
    r = reconciliar_siturq(d)
    r.to_parquet(GOLD_RECONCILIACION, index=False)
    anual = r.assign(anio=pd.to_datetime(r.periodo).dt.year).groupby(["puerto", "anio"])[
        ["pasajeros_datatur", "cruceristas_siturq"]].sum()
    anual["diferencia_pct"] = ((anual.cruceristas_siturq / anual.pasajeros_datatur - 1) * 100).round(2)
    print("DataTur vs SITUR-Q por año:\n" + anual.to_string())
    return d


In [21]:
%%modulo torre.base.silver_restmex
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Limpia el corpus de reseñas Rest-Mex 2025 (208,051 reseñas de turistas en México, con calificación de
#                    1 a 5, pueblo, estado y tipo: hotel, restaurante o atractivo) y lo guarda en Silver. PySpark lee
#                    el CSV directo (el texto trae saltos de línea dentro de las comillas).
# Por qué así:       - Es el único texto de turistas con licencia libre (CC-BY-4.0). TripAdvisor y Google no se pueden
#                      extraer (regla de oro 4).
#                    - Ojo con lo que NO cubre: en Quintana Roo solo trae Tulum (45,345), Isla Mujeres (29,826) y Bacalar
#                      (10,822). Ninguno de los 5 lugares de la campaña. Por eso sirve para saber qué molesta en los
#                      destinos llenos (multitudes, precio, sargazo), no para opinar del sur.
#                    - 178 filas idénticas en todas sus columnas (misma reseña dos veces) → se deja una. Revisado el
#                      02-oct-2026.
#                    - Nombres "QuintanaRoo" e "Isla_Mujeres" → "Quintana Roo" e "Isla Mujeres". Tipo en español.
#                    - Alternativa descartada: guardar solo Quintana Roo. El país entero sirve para comparar y es el
#                      volumen que se procesa con Spark.
# Datos de entrada:  datos/bronze/restmex/<fecha>/Rest-Mex_2025_train.csv (D5).
# Alimenta a:        Fase 8 (minería de texto: temas, sentimiento y lenguaje real para los mensajes de la campaña).

import re

from pyspark.sql import functions as F

from torre.base.entorno import RAIZ, crear_spark

BRONZE = RAIZ / "datos" / "bronze"
SILVER_RESTMEX = RAIZ / "datos" / "silver" / "restmex"

TIPOS = {"Hotel": "hotel", "Restaurant": "restaurante", "Attractive": "atractivo"}
# Papel de los pueblos de Quintana Roo en la campaña (docs/regiones/REGIONES.md).
PAPEL_QROO = {"Tulum": "referencia", "Isla Mujeres": "excluida", "Bacalar": "excluida"}


def nombre_legible(texto: str) -> str:
    """'QuintanaRoo' → 'Quintana Roo'; 'Estado_de_Mexico' → 'Estado de Mexico'; 'Baja_CaliforniaSur' → 'Baja California Sur'."""
    t = texto.replace("_", " ")
    return re.sub(r"(?<=[a-záéíóúñ])(?=[A-ZÁÉÍÓÚÑ])", " ", t).strip()


def leer_restmex(spark):
    archivo = sorted(BRONZE.glob("restmex/*/Rest-Mex_2025_train.csv"))[-1]
    df = (spark.read.option("header", True).option("multiLine", True).option("escape", '"').option("quote", '"')
          .option("encoding", "UTF-8").csv(str(archivo)))
    return df, archivo


def limpiar(df, archivo):
    legible = F.udf(nombre_legible)
    mapa_tipo = F.create_map(*[F.lit(x) for kv in TIPOS.items() for x in kv])
    mapa_papel = F.create_map(*[F.lit(x) for kv in PAPEL_QROO.items() for x in kv])
    d = (df.dropDuplicates()
         .select(F.col("Title").alias("titulo"), F.col("Review").alias("resena"),
                 F.col("Polarity").cast("double").cast("int").alias("calificacion"),
                 legible("Town").alias("pueblo"), legible("Region").alias("estado"),
                 mapa_tipo[F.col("Type")].alias("tipo"))
         .withColumn("es_qroo", F.col("estado") == "Quintana Roo")
         .withColumn("papel_campana", F.when(F.col("es_qroo"), mapa_papel[F.col("pueblo")]))
         .withColumn("n_palabras", F.size(F.split(F.trim(F.col("resena")), r"\s+")))
         .withColumn("archivo", F.lit(str(archivo.relative_to(RAIZ)))))
    return d


def construir_silver_restmex(spark=None):
    spark = spark or crear_spark("silver-restmex")
    df, archivo = leer_restmex(spark)
    crudas = df.count()
    d = limpiar(df, archivo)
    d.write.mode("overwrite").partitionBy("estado").parquet(str(SILVER_RESTMEX))
    final = spark.read.parquet(str(SILVER_RESTMEX))
    n = final.count()
    print(f"Rest-Mex Silver: {n:,} reseñas ({crudas - n} idénticas quitadas de {crudas:,})")
    final.filter("es_qroo").groupBy("pueblo", "tipo").count().orderBy("pueblo", "tipo").show(truncate=False)
    nulos = final.filter(F.col("tipo").isNull() | F.col("calificacion").isNull()).count()
    if nulos:
        raise ValueError(f"Rest-Mex: {nulos} reseñas sin tipo o sin calificación; revisar la lectura del CSV")
    return final


In [22]:
import time
import torre.base as b
trabajos = [("SITUR-Q", b.silver_siturq.construir_silver_siturq), ("DataTur ocupación", b.silver_datatur_ocupacion.construir_silver_ocupacion),
            ("DENUE (6.1 millones de negocios)", b.silver_denue.construir_silver_denue), ("INAH", b.silver_inah.construir_silver_inah),
            ("Censo 2020", b.silver_iter.construir_silver_iter), ("Clima", b.silver_clima.construir_silver_clima),
            ("Huracanes", b.silver_huracanes.construir_silver_huracanes), ("Tipo de cambio", b.silver_fred.construir_silver_fred),
            ("Nacionalidades", b.silver_nacionalidad.construir_silver_nacionalidad), ("Vuelos AFAC", b.silver_afac.construir_silver_afac),
            ("Cruceros", b.silver_cruceros.construir_silver_cruceros), ("Reseñas Rest-Mex", b.silver_restmex.construir_silver_restmex)]
for nombre, trabajo in trabajos:
    if not BRONZE_COMPLETO:
        print(f"· {nombre}: se usa la tabla limpia que ya viene"); continue
    t0 = time.time(); trabajo(spark)
    print(f"✓ {nombre}: limpio en {time.time() - t0:,.1f} s")

✓ SITUR-Q: limpio en 27.6 s


✓ DataTur ocupación: limpio en 52.1 s


✓ DENUE (6.1 millones de negocios): limpio en 75.9 s


INAH Silver: 71,278 filas (283 duplicadas quitadas); Quintana Roo: 3,594 filas, 16 sitios
Visitantes 2025 por región y papel:
papel_campana  region_campana           
referencia     Tulum                        1031443
excluida       Chacchoben                    237039
retirada       Cobá + Punta Laguna           191815
excluida       Cozumel                       143541
referencia     Cancún                         89940
promovida      Ruta arqueológica del sur      66628
               Bahía Calderitas–Oxtankah      11017
referencia     Riviera Maya                     238
retirada       Muyil                              0
✓ INAH: limpio en 51.4 s


ITER Silver: 2,243 filas; 2,207 localidades; población estatal 1,857,985; 1,627 localidades con algún dato reservado por INEGI
                                         poblacion  viviendas_habitadas  viviendas_con_agua  viviendas_con_drenaje
papel_campana region_campana                                                                                      
promovida     Bahía Calderitas–Oxtankah       5551                 1610                1586                   1587
              Chetumal                      169028                51399               50696                  50816
              Laguna Milagros–Xul-Ha          4182                 1243                1217                   1231
              Maya Ka'an + Kantemó           44388                11543               11307                  10960
              Ruta arqueológica del sur       6098                 1719                1674                   1636
referencia    Cancún                        888797               280

...

Cruceros Silver: 3,895 filas; Quintana Roo: 762 filas, 4 puertos sin cruceros en toda la serie


DataTur vs SITUR-Q por año:
               pasajeros_datatur  cruceristas_siturq  diferencia_pct
puerto   anio                                                       
Cozumel  2019            4569853           4772311.0            4.43
         2020            1132101           1283181.0           13.35
         2021             650107            850866.0           30.88
         2022            2938153           3378261.0           14.98
         2023            4076976           4283003.0            5.05
         2024            4619126           4598918.0           -0.44
         2025            4724255           4915242.0            4.04
         2026            3079763           3329173.0            8.10
Mahahual 2019            1604435           2164161.0           34.89
         2020             455179            598431.0           31.47
         2021             343082            427990.0           24.75
         2022            1229864           1525106.0           24.01
      

Rest-Mex Silver: 207,873 reseñas (178 idénticas quitadas de 208,051)


+------------+-----------+-----+
|pueblo      |tipo       |count|
+------------+-----------+-----+
|Bacalar     |atractivo  |2162 |
|Bacalar     |hotel      |2439 |
|Bacalar     |restaurante|6221 |
|Isla Mujeres|atractivo  |7286 |
|Isla Mujeres|hotel      |8853 |
|Isla Mujeres|restaurante|13686|
|Tulum       |atractivo  |23988|
|Tulum       |hotel      |11999|
|Tulum       |restaurante|9353 |
+------------+-----------+-----+



✓ Reseñas Rest-Mex: limpio en 40.7 s


In [23]:
# Las 14 tablas limpias que quedaron (leídas con Spark)
filas = []
for carpeta in sorted((RAIZ / "datos" / "silver").iterdir()):
    if carpeta.is_dir():
        n = spark.read.parquet(str(carpeta)).count()
        mb = sum(f.stat().st_size for f in carpeta.rglob("*.parquet")) / 1e6
        filas.append({"tabla": carpeta.name, "renglones": n, "MB": round(mb, 1)})
pd.DataFrame(filas).sort_values("renglones", ascending=False)

,tabla,renglones,MB
5,denue,6138075,291.6
2,clima_horario,542784,8.1
11,nacionalidad,521363,1.7
1,clima_diario,224232,5.3
12,restmex,207873,52.3
9,inah,71278,0.9
8,huracanes,55524,2.0
4,datatur_ocupacion,23341,0.7
0,afac,19277,0.2
6,fred_diario,8575,0.2


## 5. Spark sobre la tabla grande: los negocios del país
La tabla del DENUE tiene más de 6 millones de negocios y está **partida por estado**: para leer Quintana Roo, Spark abre
solo la carpeta `cve_ent=23` (*partition pruning*). Las transformaciones se planean completas y se ejecutan hasta que se
pide un resultado (*evaluación perezosa*).

In [24]:
from pyspark.sql import functions as F
denue = spark.read.parquet(str(RAIZ / "datos" / "silver" / "denue"))
print(f"Negocios en el país: {denue.count():,} · turísticos: {denue.where('es_turistico').count():,}")
(denue.where("es_qroo and es_turistico").groupBy("municipio").pivot("categoria_turistica").count()
 .fillna(0).orderBy(F.desc("Alimentos y bebidas")).toPandas())

Negocios en el país: 6,138,075 · turísticos: 875,667


,municipio,Agencias de viajes,Alimentos y bebidas,Alojamiento,Esparcimiento,Museos y sitios históricos,Transporte turístico
0,Benito Juárez,177,4429,277,363,13,34
1,Solidaridad,105,1685,256,129,23,6
2,Othón P. Blanco,16,1558,118,135,8,1
3,Cozumel,34,854,77,169,12,2
4,Tulum,21,703,288,36,31,1
5,Felipe Carrillo Puerto,3,356,17,22,4,0
6,Isla Mujeres,9,273,75,24,4,0
7,Bacalar,6,259,136,18,2,1
8,Puerto Morelos,5,252,32,26,4,0
9,Lázaro Cárdenas,6,234,153,11,0,1


In [25]:
# Cuántos negocios turísticos hay en cada estado (las 10 con más)
(denue.where("es_turistico").groupBy("entidad").agg(F.count("*").alias("negocios_turisticos"))
 .orderBy(F.desc("negocios_turisticos")).limit(10).toPandas())

,entidad,negocios_turisticos
0,México,103264
1,Ciudad de México,62960
2,Jalisco,59038
3,Veracruz de Ignacio de la Llave,57731
4,Puebla,52457
5,Oaxaca,42016
6,Michoacán de Ocampo,40058
7,Guanajuato,39525
8,Chiapas,35432
9,Nuevo León,30706


## 6. El almacén: DuckDB y el modelo estrella
DuckDB es una base de datos en un solo archivo: lee las tablas de plata y oro sin copiarlas y responde en milisegundos, sin
servidor ni internet. La tabla de **hechos** tiene un renglón por lugar, mes y variable; a sus lados están las tablas de
lugares y de meses. **Un hueco no tiene renglón**: nunca se rellena con cero.

### El código del almacén y del diccionario
Cada celda de código de abajo es un archivo del paquete `torre`, completo y sin cambios (con su encabezado: qué hace, por qué así, de qué datos sale y a qué decisión alimenta). Al correrla, la celda queda registrada como ese módulo y las celdas siguientes lo usan.

- **`torre.base.almacen`** — Arma el almacén de consulta del proyecto en DuckDB (datos/gold/torre.duckdb): 1. Una vista por cada tabla de Silver (silver_<tabla>) y por cada salida de Gold (gold_<tabla>). 2. El modelo estrella del plan (Fase 2, punto 2): - dim_lugar: los lugares con su papel (promovido, referencia o comparación). - dim_tiempo: un renglón por mes de 2012 a 2027. - hechos_mes: una medida por lugar y mes, en formato largo (lugar, periodo, variable, valor, fuente), para cruzar fuentes sin tener que saber en qué archivo está cada una.
- **`torre.base.diccionario`** — Genera el diccionario de datos (datos/DICCIONARIO.md) leyendo el almacén DuckDB: por cada tabla de Silver, de Gold y del modelo estrella da su descripción, renglones, y por cada columna su tipo, % de vacíos, un ejemplo real y qué significa.

In [26]:
%%modulo torre.base.almacen
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Arma el almacén de consulta del proyecto en DuckDB (datos/gold/torre.duckdb):
#                    1. Una vista por cada tabla de Silver (silver_<tabla>) y por cada salida de Gold (gold_<tabla>).
#                    2. El modelo estrella del plan (Fase 2, punto 2):
#                       - dim_lugar: los lugares con su papel (promovido, referencia o comparación).
#                       - dim_tiempo: un renglón por mes de 2012 a 2027.
#                       - hechos_mes: una medida por lugar y mes, en formato largo (lugar, periodo, variable, valor,
#                         fuente), para cruzar fuentes sin tener que saber en qué archivo está cada una.
# Por qué así:       - Spark limpia y escribe (Bronze → Silver), pero para responder una pregunta en milisegundos a la
#                      página o al equipo no hace falta levantar Spark: DuckDB lee los mismos Parquet sin copiarlos.
#                    - Vistas y no copias: si se reconstruye Silver, el almacén ve el dato nuevo sin volver a cargarlo.
#                      Solo las tres tablas de la estrella se materializan, porque son chicas y se consultan mucho.
#                    - Formato largo en hechos_mes: cada fuente aporta las variables que tiene. Un hueco simplemente no
#                      tiene renglón; nunca se rellena con cero.
#                    - Alternativa descartada: una base de datos con servidor (PostgreSQL). Exige instalar y levantar un
#                      servicio, y el proyecto debe correr sin internet en cualquier computadora.
# Datos de entrada:  datos/silver/* (Parquet particionado) y datos/gold/*.parquet.
# Alimenta a:        La página y el informe (consultas rápidas y rastreables), el diccionario de datos
#                    (torre.base.diccionario) y la Fase 9 (endpoint de consulta de solo lectura).

from pathlib import Path

import duckdb

from torre.base.entorno import RAIZ

SILVER = RAIZ / "datos" / "silver"
GOLD = RAIZ / "datos" / "gold"
ALMACEN = GOLD / "torre.duckdb"

# Variables del panel del Radar que entran a hechos_mes, con su fuente oficial.
VARIABLES_PANEL = {
    "llegadas_tren": "SITUR-Q (Tren Maya)",
    "cruceristas": "SITUR-Q",
    "cruces_belice": "SITUR-Q",
    "visitantes_inah": "INAH (DataTur BdINAH)",
    "ocupacion_pct": "SITUR-Q o DataTur (una sola por lugar, Fase 4)",
    "cuartos": "SITUR-Q",
    "hoteles": "SITUR-Q",
}
# Lugares de referencia (regla de oro 9). Los 5 lugares salen del panel (es_de_los_5).
REFERENCIA = ("Cancún", "Playa del Carmen", "Tulum")


def tablas_silver() -> list[Path]:
    return sorted(p for p in SILVER.iterdir() if p.is_dir() and any(p.rglob("*.parquet")))


def tablas_gold() -> list[Path]:
    return sorted(GOLD.glob("*.parquet"))


def _ruta(p: Path) -> str:
    return p.as_posix().replace("'", "''")


def crear_vistas(con: duckdb.DuckDBPyConnection) -> None:
    for p in tablas_silver():
        con.execute(f"CREATE OR REPLACE VIEW silver_{p.name} AS SELECT * FROM "
                    f"read_parquet('{_ruta(p)}/**/*.parquet', hive_partitioning = true, union_by_name = true)")
    for p in tablas_gold():
        con.execute(f"CREATE OR REPLACE VIEW gold_{p.stem} AS SELECT * FROM read_parquet('{_ruta(p)}')")


def crear_estrella(con: duckdb.DuckDBPyConnection) -> None:
    con.execute("""
        CREATE OR REPLACE TABLE dim_tiempo AS
        SELECT CAST(p AS DATE) AS periodo, year(p) AS anio, month(p) AS mes, quarter(p) AS trimestre
        FROM range(DATE '2012-01-01', DATE '2028-01-01', INTERVAL 1 MONTH) t(p)""")
    con.execute(f"""
        CREATE OR REPLACE TABLE dim_lugar AS
        SELECT lugar, bool_or(es_de_los_5) AS es_de_los_5,
               CASE WHEN bool_or(es_de_los_5) THEN 'promovido'
                    WHEN lugar IN {REFERENCIA} THEN 'referencia' ELSE 'comparación' END AS papel
        FROM gold_radar_panel_mensual GROUP BY lugar""")
    partes = [f"SELECT lugar, CAST(periodo AS DATE) AS periodo, '{v}' AS variable, CAST({v} AS DOUBLE) AS valor, "
              f"'{f}' AS fuente FROM gold_radar_panel_mensual WHERE {v} IS NOT NULL" for v, f in VARIABLES_PANEL.items()]
    # Extranjeros por avión: el aeropuerto se asigna a su lugar (Chetumal, Cancún, Tulum, Cozumel).
    partes.append("""SELECT lugar_campana, CAST(periodo AS DATE), 'llegadas_extranjeros_avion',
                            CAST(sum(llegadas_extranjeros) AS DOUBLE), 'DataTur BD_Nacionalidad (UPM)'
                     FROM silver_nacionalidad WHERE es_qroo GROUP BY ALL""")
    # Pasajeros de crucero según DataTur, solo de los puertos que sí reciben cruceros.
    partes.append("""SELECT puerto, CAST(periodo AS DATE), 'pasajeros_crucero_datatur', CAST(pasajeros AS DOUBLE),
                            'DataTur BaseDatosCruceros'
                     FROM silver_cruceros WHERE es_qroo AND NOT puerto_sin_cruceros_flag""")
    con.execute("CREATE OR REPLACE TABLE hechos_mes AS " + "\nUNION ALL\n".join(partes))
    # Un lugar que aparece en los hechos y no en dim_lugar (p. ej., Mahahual por cruceros) entra como comparación.
    con.execute("""INSERT INTO dim_lugar SELECT DISTINCT h.lugar, false, 'comparación' FROM hechos_mes h
                   WHERE h.lugar NOT IN (SELECT lugar FROM dim_lugar)""")


def construir_almacen(ruta: Path = ALMACEN) -> Path:
    if ruta.exists():
        ruta.unlink()
    with duckdb.connect(str(ruta)) as con:
        crear_vistas(con)
        crear_estrella(con)
        n_s, n_g = len(tablas_silver()), len(tablas_gold())
        n_h = con.execute("SELECT count(*) FROM hechos_mes").fetchone()[0]
        print(f"Almacén DuckDB: {n_s} vistas de Silver, {n_g} de Gold; hechos_mes con {n_h:,} renglones")
        print(con.execute("""SELECT variable, count(DISTINCT lugar) AS lugares, min(periodo) AS desde, max(periodo) AS hasta,
                                    count(*) AS renglones FROM hechos_mes GROUP BY variable ORDER BY variable""").df()
              .to_string(index=False))
    return ruta


def consultar(sql: str):
    """Consulta de SOLO LECTURA al almacén (para la página o el equipo). Devuelve un DataFrame."""
    with duckdb.connect(str(ALMACEN), read_only=True) as con:
        return con.execute(sql).df()


In [27]:
%%modulo torre.base.diccionario
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Genera el diccionario de datos (docs/datos/DICCIONARIO.md) leyendo el almacén DuckDB: por cada tabla
#                    de Silver, de Gold y del modelo estrella da su descripción, renglones, y por cada columna su tipo,
#                    % de vacíos, un ejemplo real y qué significa.
# Por qué así:       - Generado, no escrito a mano: si cambia una tabla, el diccionario cambia con ella y nunca queda viejo.
#                      Lo que sí se escribe a mano es el SIGNIFICADO (abajo, en TABLAS y COLUMNAS).
#                    - Toda columna de Silver debe tener significado; si falta, el proceso se detiene (lo vigila
#                      tests/test_silver_fase2.py). En Gold, las columnas que no están descritas remiten a la nota de
#                      decisión de su fase, donde se explica el modelo.
# Datos de entrada:  datos/gold/torre.duckdb (torre.base.almacen).
# Alimenta a:        El informe técnico (criterio 2 de la rúbrica: estructura y documenta los datos) y al equipo.

import duckdb

from torre.base.almacen import ALMACEN
from torre.base.entorno import RAIZ

SALIDA = RAIZ / "datos" / "DICCIONARIO.md"

# ---------- Qué es cada tabla ----------
TABLAS = {
    # Silver (limpio y tipado)
    "silver_afac": "Pasajeros por aerolínea en México, por mes (AFAC vía DataTur, 2016 → jul-2026). Nacional.",
    "silver_clima_diario": "Clima diario de 8 puntos de Quintana Roo (Open-Meteo, 1950 → 2026).",
    "silver_clima_horario": "Clima por hora de 8 puntos (Open-Meteo, 2019 → 2026), con hora local.",
    "silver_cruceros": "Arribos y pasajeros de crucero por puerto y mes (DataTur, 2016 → jul-2026).",
    "silver_datatur_ocupacion": "Ocupación hotelera semanal y mensual por centro turístico (DataTur, 2022 → 2026).",
    "silver_denue": "Negocios del país con giro, tamaño y coordenadas (DENUE INEGI), con su categoría turística.",
    "silver_fred_diario": "Pesos por dólar por día (FRED, DEXMXUS).",
    "silver_fred_mensual": "Pesos por dólar promedio del mes e inflación de EE. UU. (FRED).",
    "silver_huracanes": "Posiciones de todas las tormentas del Atlántico cada 6 horas (HURDAT2, NOAA, 1851 → 2025).",
    "silver_inah": "Visitantes a museos y zonas arqueológicas por mes y tipo (INAH vía DataTur, 2016 → 2026).",
    "silver_iter": "Población y viviendas por localidad (Censo 2020, ITER Quintana Roo).",
    "silver_nacionalidad": "Llegadas de extranjeros por avión, por aeropuerto, país y sexo (UPM vía DataTur, 2012 → jul-2026).",
    "silver_restmex": "Reseñas de turistas con calificación de 1 a 5 (Rest-Mex 2025, CC-BY-4.0).",
    "silver_siturq": "Indicadores turísticos de Quintana Roo por destino y mes (SITUR-Q, API pública).",
    # Modelo estrella
    "dim_lugar": "Dimensión de lugares: cada lugar con su papel (promovido, referencia o comparación).",
    "dim_tiempo": "Dimensión de tiempo: un renglón por mes, de 2012 a 2027.",
    "hechos_mes": "Hechos: una medida por lugar, mes y variable, con su fuente. Un hueco no tiene renglón.",
    # Gold (salidas de los modelos)
    "gold_criterios_regiones": "Fase 3. Tabla de los 5 criterios por región (sargazo, cierres, saturación, fragilidad, datos).",
    "gold_lugares_clasificados": "Planeador. Negocios del DENUE clasificados por léxico (hospedaje, comida, qué hacer).",
    "gold_lugares_recomendados": "Planeador. Negocios recomendados por lugar y momento del día.",
    "gold_pronostico_backtest": "Fase 5. Cada pronóstico del origen móvil contra el dato real.",
    "gold_pronostico_calendario": "Fase 5. Calendario lugar × mes: temporada alta, tormenta, lluvia y otro mes u otro lugar.",
    "gold_pronostico_cobertura": "Fase 5. Qué tanto el rango del 90 % contuvo al dato real, por modelo y horizonte.",
    "gold_pronostico_eleccion": "Fase 5. Modelo elegido por serie (menor error con rango confiable).",
    "gold_pronostico_escenarios": "Fase 5. Monte Carlo: escenarios malo/probable/bueno por mes y riesgo de rebasar capacidad.",
    "gold_pronostico_escenarios_anual": "Fase 5. Escenarios sumados al año, por supuesto de golpe de tormenta.",
    "gold_pronostico_forma_anio": "Fase 5. Forma del año: índice estacional de cada mes por serie.",
    "gold_pronostico_fuerza_estacional": "Fase 5. Fuerza estacional por serie (clásica y STL).",
    "gold_pronostico_mes": "Fase 5. Pronóstico de los próximos 12 meses con su rango del 90 % (_est).",
    "gold_pronostico_metricas": "Fase 5. Errores (MAE, MAPE) de cada modelo por serie.",
    "gold_pronostico_poisson_tormentas": "Fase 5. Probabilidad de tormenta por mes (Poisson, 1966–2025).",
    "gold_pronostico_sensibilidad": "Fase 5. Efecto de lluvia y dólar en cada serie, con su p-valor.",
    "gold_pronostico_series": "Fase 5. Series mensuales que se pronostican, con huecos y tramos marcados.",
    "gold_radar_clusters_centros": "Fase 4. Agrupamiento Ward de los centros turísticos del país (DataTur).",
    "gold_radar_estado": "Fase 4. Índice de presión y estado (tranquilo/concurrido/saturado) por lugar y mes.",
    "gold_radar_indice_comparable": "Fase 4. Índice comparable (mismas medidas en todos los lugares).",
    "gold_radar_markov_backtest": "Fase 4. Prueba de la cadena de Markov semanal contra la persistencia.",
    "gold_radar_markov_matriz": "Fase 4. Matriz de transición semanal entre estados (norte, DataTur).",
    "gold_radar_markov_riesgo": "Fase 4. Probabilidad de cada estado a 1–8 semanas.",
    "gold_radar_modelos": "Fase 4. Comparación de clasificadores del estado del mes siguiente.",
    "gold_radar_panel_mensual": "Fase 4. Panel lugar × mes con todas las variables de presión.",
    "gold_radar_prediccion": "Fase 4. Estado esperado del mes siguiente por lugar (_est).",
    "gold_radar_sesgo": "Fase 4. Aciertos del modelo en los 5 lugares contra el norte (sesgo).",
    "gold_reconciliacion_cruceros": "Fase 2. Cruceristas: DataTur contra SITUR-Q, mes a mes, en Cozumel y Mahahual.",
    "gold_presupuesto_plan": "Fase 6. Pesos y visitantes esperados (_est) por mes, lugar y canal del plan óptimo.",
    "gold_presupuesto_pausas": "Fase 6. Si el anuncio sigue encendido en cada escenario (malo, probable, bueno), donde hay gasto.",
    "gold_presupuesto_reglas": "Fase 6. Visitantes que cuesta cada regla (resolviendo el modelo sin ella).",
    "gold_presupuesto_precios_sombra": "Fase 6. Precio sombra y holgura del presupuesto, la equidad y los topes por canal.",
    "gold_presupuesto_pareto": "Fase 6. Frontera de Pareto: visitantes esperados según la ocupación máxima permitida.",
    "gold_presupuesto_sensibilidad": "Fase 6. El modelo resuelto con cada supuesto movido (conversión, clic, tormentas, presupuesto).",
    "gold_envivo_senales": "Fase 7. Las cuatro señales de cada semana: norte, tormentas, clima raro y llegadas.",
    "gold_envivo_decisiones": "Fase 7. Qué hizo la Torre cada semana en cada lugar del sur y cuánto dinero gastó o guardó.",
    "gold_envivo_norte": "Fase 7. Semanas con el norte saturado y si se encendió \"¿Ibas al norte?\".",
    "gold_campana_aspectos": "Fase 8. Temas de las reseñas: % que los menciona y riesgo relativo de reseña mala.",
    "gold_campana_reglas": "Fase 8. Reglas de asociación (Apriori) que terminan en reseña mala o de 5 estrellas.",
    "gold_campana_palabras": "Fase 8. Palabras que distinguen reseñas de 5 estrellas y malas (log-odds).",
}

# ---------- Qué significa cada columna (las que se repiten van una sola vez) ----------
COLUMNAS = {
    "anio": "Año.", "mes": "Mes (1–12).", "periodo": "Primer día del mes.", "fecha": "Fecha.",
    "archivo": "Archivo crudo de Bronze de donde salió el renglón (rastreo, regla de oro 2).",
    "es_qroo": "Verdadero si es de Quintana Roo.",
    "papel_campana": "Papel en la campaña: promovida, referencia, retirada o excluida (docs/regiones/REGIONES.md).",
    "region_campana": "Región de la campaña a la que pertenece.", "lugar_campana": "Lugar de la campaña al que pertenece.",
    "lat": "Latitud (grados).", "lon": "Longitud (grados).", "latitud": "Latitud (grados).", "longitud": "Longitud (grados).",
    "estado": "Entidad federativa.", "sin_dato_flag": "Verdadero si la fuente no trae dato (hueco; no se rellena).",
    "fuente": "Fuente oficial.", "fuente_tipo": "Tipo de dato de la fuente (observado o reanálisis).",
    "cve_mun": "Clave INEGI del municipio.", "municipio": "Municipio.", "cve_loc": "Clave INEGI de la localidad.",
    "localidad": "Localidad.", "tipo_fila": "Tipo de renglón en la fuente (dato, total o nota).",
    # AFAC
    "tipo_vuelo": "nacional o internacional.", "servicio": "regular o fletamento.",
    "region_aerolinea": "Región de origen de la aerolínea.", "aerolinea": "Aerolínea.",
    "pasajeros": "Pasajeros (AFAC: transportados; cruceros: pasajeros que llegaron en el mes).",
    "sumada_flag": "Verdadero si el renglón suma dos cifras de la misma etiqueta (Virgin America + Alaska, 2016–ene-2018).",
    # Clima
    "punto": "Punto de clima (8 puntos de Quintana Roo).", "temp_max_c": "Temperatura máxima del día (°C).",
    "lluvia_mm": "Lluvia (mm).", "viento_max_kmh": "Viento máximo del día (km/h).",
    "fecha_hora_utc": "Fecha y hora en UTC, como la entrega la fuente.", "fecha_hora_local": "Fecha y hora local (UTC − 5 h).",
    "temp_c": "Temperatura (°C).", "viento_kmh": "Viento (km/h).",
    # Cruceros
    "puerto": "Puerto de cruceros.", "arribos": "Barcos que llegaron en el mes.", "litoral": "Pacífico o Golfo-Caribe.",
    "puerto_sin_cruceros_flag": "Verdadero si el puerto tiene 0 pasajeros en toda la serie (catálogo sin cruceros).",
    # DataTur ocupación
    "centro": "Centro turístico (nombre limpio).", "centro_crudo": "Centro turístico tal como viene en el archivo.",
    "cuartos_disponibles": "Cuartos disponibles promedio diario.", "cuartos_ocupados": "Cuartos ocupados promedio diario.",
    "no_disponible_flag": "Verdadero si la fuente marca el dato como no disponible.",
    "nota_al_pie": "Nota al pie de la fuente (bandera de comparabilidad).", "ocupacion_pct": "Ocupación hotelera (%).",
    "semana": "Semana del año (ISO).", "n_versiones": "Cuántas veces se publicó la misma semana o mes.",
    "revisado_flag": "Verdadero si la cifra cambió entre publicaciones (se conserva la última).",
    "ocupacion_calc_pct": "Ocupación recalculada como ocupados ÷ disponibles × 100 (comprobación).",
    "frecuencia": "semanal o mensual.",
    # DENUE
    "id": "Identificador del negocio en el DENUE.", "nom_estab": "Nombre del establecimiento.",
    "codigo_act": "Código SCIAN de la actividad.", "nombre_act": "Actividad económica.",
    "per_ocu": "Personal ocupado (rango, como lo publica INEGI).", "tipoUniEco": "Tipo de unidad económica (fija o semifija).",
    "entidad": "Entidad federativa.", "cod_postal": "Código postal.", "fecha_alta": "Fecha de alta en el DENUE.",
    "scian_3": "Subsector SCIAN (3 dígitos).", "categoria_turistica": "Categoría turística (hospedaje, alimentos, etc.).",
    "es_turistico": "Verdadero si el giro es turístico.", "personas_min": "Límite inferior del rango de personal.",
    "personas_max": "Límite superior del rango de personal.",
    "coordenadas_flag": "Verdadero si las coordenadas caen fuera de su municipio o faltan.", "cve_ent": "Clave INEGI de la entidad.",
    # FRED
    "pesos_por_dolar": "Pesos mexicanos por dólar.", "n_dias_observados": "Días con cotización en el mes.",
    "n_dias_sin_dato": "Días hábiles sin cotización (feriados).",
    "mes_incompleto_flag": "Verdadero si el mes aún no termina.", "inflacion_eeuu_indice": "Índice de precios al consumidor de EE. UU.",
    # Huracanes
    "id_tormenta": "Identificador de la tormenta (HURDAT2).", "nombre": "Nombre (tormenta o sitio INAH).",
    "hora_utc": "Hora UTC (HHMM).", "marca": "Marca especial del registro (p. ej., L = toca tierra).",
    "estado_sistema": "Tipo de sistema (HU huracán, TS tormenta tropical, etc.).", "viento_kt": "Viento sostenido (nudos).",
    "presion_mb": "Presión central (mb); vacío si la fuente trae −999.", "km_chetumal": "Distancia a Chetumal (km).",
    "dentro_radio_flag": "Verdadero si pasa a ≤ 200 km de Chetumal.", "tormenta_o_huracan_flag": "Verdadero si el viento es ≥ 34 nudos.",
    "era_satelital_flag": "Verdadero desde 1966 (era satelital).", "afecta_sur_flag": "Verdadero si cumple las tres reglas anteriores.",
    "formato_irregular_flag": "Verdadero si la línea original venía mal formada.",
    # INAH
    "clasificacion": "Museo o zona arqueológica.", "tipo_visitante": "nacional o extranjero.",
    "visitantes": "Visitantes en el mes.", "sin_visitantes_flag": "Verdadero si el mes tiene 0 visitantes (suele ser cierre).",
    # ITER
    "poblacion": "Habitantes (Censo 2020).", "viviendas_habitadas": "Viviendas particulares habitadas.",
    "viviendas_con_agua": "Viviendas con agua entubada.", "viviendas_con_drenaje": "Viviendas con drenaje.",
    "viviendas_con_luz": "Viviendas con electricidad.", "reservado_flag": "Verdadero si INEGI reserva el dato por confidencialidad.",
    # Nacionalidad
    "aeropuerto": "Aeropuerto de llegada.", "pais": "País de nacionalidad.", "region_mundo": "Región del mundo.",
    "sexo": "hombre, mujer o no disponible.", "llegadas_extranjeros": "Extranjeros que llegaron por avión en el mes.",
    # Rest-Mex
    "titulo": "Título de la reseña.", "resena": "Texto de la reseña.", "calificacion": "Calificación de 1 a 5.",
    "pueblo": "Pueblo reseñado.", "tipo": "hotel, restaurante o atractivo.", "n_palabras": "Palabras de la reseña.",
    # SITUR-Q
    "unidad": "Destino o zona de SITUR-Q.", "tipo_unidad": "destino, zona o zona especial.",
    "variable": "Variable medida.", "valor": "Valor de la variable.", "n_registros_fuente": "Registros de la fuente que se sumaron.",
    "hueco_flag": "Verdadero si la fuente trae 0 o vacío donde debería haber dato (hueco declarado).",
    "indicador": "Indicador de SITUR-Q (ocupación, cruceristas, Tren Maya, etc.).",
    # Estrella
    "lugar": "Lugar.", "es_de_los_5": "Verdadero si es una de las 5 regiones de la campaña.",
    "papel": "promovido, referencia o comparación.", "trimestre": "Trimestre (1–4).",
    # Reconciliación
    "pasajeros_datatur": "Pasajeros de crucero según DataTur.", "cruceristas_siturq": "Cruceristas según SITUR-Q.",
    "diferencia": "SITUR-Q − DataTur.", "diferencia_pct": "Diferencia como % de DataTur.",
}


def _ejemplo(con, tabla: str, columna: str) -> str:
    v = con.execute(f'SELECT "{columna}" FROM {tabla} WHERE "{columna}" IS NOT NULL LIMIT 1').fetchone()
    if v is None:
        return "—"
    t = str(v[0]).replace("|", "/").replace("\n", " ")
    return t[:40] + ("…" if len(t) > 40 else "")


def describir(con, tabla: str) -> tuple[int, list[dict]]:
    n = con.execute(f"SELECT count(*) FROM {tabla}").fetchone()[0]
    filas = []
    for col, tipo, *_ in con.execute(f"DESCRIBE {tabla}").fetchall():
        vacios = con.execute(f'SELECT avg(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) FROM {tabla}').fetchone()[0] or 0
        filas.append({"columna": col, "tipo": tipo, "vacios_pct": 100 * vacios, "ejemplo": _ejemplo(con, tabla, col),
                      "significado": COLUMNAS.get(col)})
    return n, filas


def faltantes_silver(con) -> list[str]:
    """Columnas de Silver sin significado escrito (debe quedar vacía)."""
    falta = []
    for (t,) in con.execute("SELECT view_name FROM duckdb_views() WHERE view_name LIKE 'silver_%'").fetchall():
        falta += [f"{t}.{c}" for c, *_ in con.execute(f"DESCRIBE {t}").fetchall() if c not in COLUMNAS]
    return falta


def generar() -> str:
    with duckdb.connect(str(ALMACEN), read_only=True) as con:
        falta = faltantes_silver(con)
        if falta:
            raise ValueError(f"Columnas de Silver sin significado: {falta}")
        nombres = [r[0] for r in con.execute(
            "SELECT view_name FROM duckdb_views() WHERE NOT internal UNION ALL "
            "SELECT table_name FROM duckdb_tables()").fetchall()]
        orden = ([n for n in sorted(nombres) if n.startswith("silver_")] + ["dim_lugar", "dim_tiempo", "hechos_mes"]
                 + [n for n in sorted(nombres) if n.startswith("gold_")])
        sin_tabla = [n for n in orden if n not in TABLAS]
        if sin_tabla:
            raise ValueError(f"Tablas sin descripción: {sin_tabla}")
        md = ["# Diccionario de datos — Torre del Caribe", "",
              "Autor: **Brandon Uriel García Sánchez** · Generado por `python -m torre.base.diccionario` desde el almacén "
              "`datos/gold/torre.duckdb`. **No se edita a mano**: el significado de cada columna vive en "
              "`backend/torre/base/diccionario.py`.", "",
              "Capas: **Silver** = limpio y tipado (una vista por tabla) · **Estrella** = dim_lugar, dim_tiempo, hechos_mes "
              "· **Gold** = salidas de los modelos (su método está en la nota de decisión de su fase y en "
              "`docs/metodologia/ECUACIONES.md`). Sufijos: `_est` estimado, `_flag` bandera.", "",
              "| Tabla | Renglones | Qué es |", "|---|---:|---|"]
        detalle = []
        for t in orden:
            n, filas = describir(con, t)
            md.append(f"| [`{t}`](#{t}) | {n:,} | {TABLAS[t]} |")
            detalle += ["", f"## {t}", "", f"{TABLAS[t]} **{n:,} renglones.**", "",
                        "| Columna | Tipo | Vacíos | Ejemplo | Significado |", "|---|---|---:|---|---|"]
            for f in filas:
                s = f["significado"] or "Ver la nota de decisión de su fase."
                detalle.append(f"| `{f['columna']}` | {f['tipo']} | {f['vacios_pct']:.1f} % | {f['ejemplo']} | {s} |")
    texto = "\n".join(md + detalle) + "\n"
    SALIDA.write_text(texto, encoding="utf-8")
    print(f"Diccionario: {len(orden)} tablas → {SALIDA.relative_to(RAIZ)}")
    return texto


In [28]:
from torre.base import almacen
almacen.construir_almacen()
almacen.consultar("""SELECT l.lugar, l.papel, count(*) AS renglones, min(h.periodo) AS desde, max(h.periodo) AS hasta
                     FROM hechos_mes h JOIN dim_lugar l USING (lugar) GROUP BY ALL ORDER BY renglones DESC""")

Almacén DuckDB: 14 vistas de Silver, 39 de Gold; hechos_mes con 3,287 renglones
                  variable  lugares      desde      hasta  renglones
               cruceristas        2 2022-01-01 2026-07-01        110
             cruces_belice        1 2022-01-01 2026-06-01         54
                   cuartos       12 2022-01-01 2026-07-01        660
                   hoteles       12 2022-01-01 2026-07-01        660
llegadas_extranjeros_avion        4 2012-01-01 2026-07-01        498
             llegadas_tren        8 2024-01-01 2026-07-01        199
             ocupacion_pct       12 2022-01-01 2026-07-01        412
 pasajeros_crucero_datatur        2 2016-01-01 2026-07-01        254
           visitantes_inah        8 2022-01-01 2026-07-01        440


,lugar,papel,renglones,desde,hasta
0,Cozumel,comparación,577,2012-01-01,2026-07-01
1,Cancún,referencia,426,2012-01-01,2026-07-01
2,Chetumal,promovido,338,2012-02-01,2026-07-01
3,Mahahual,comparación,304,2016-01-01,2026-07-01
4,Tulum,referencia,254,2022-01-01,2026-07-01
5,Playa del Carmen,referencia,249,2022-01-01,2026-07-01
6,Costa Mujeres,comparación,201,2022-01-01,2026-07-01
7,Bacalar,comparación,197,2022-01-01,2026-07-01
8,Puerto Morelos,comparación,175,2022-01-01,2026-07-01
9,Isla Mujeres,comparación,165,2022-01-01,2026-07-01


## 7. El diccionario de datos
Se genera solo, desde el almacén: cada columna con su tipo, su porcentaje de vacíos, un ejemplo real y su significado. Si
una columna no tiene significado escrito, el programa se detiene. Queda en `datos/DICCIONARIO.md`.

In [29]:
from torre.base import diccionario
texto = diccionario.generar()
print(texto[:1500])

Diccionario: 56 tablas → datos\DICCIONARIO.md
# Diccionario de datos — Torre del Caribe

Autor: **Brandon Uriel García Sánchez** · Generado por `python -m torre.base.diccionario` desde el almacén `datos/gold/torre.duckdb`. **No se edita a mano**: el significado de cada columna vive en `backend/torre/base/diccionario.py`.

Capas: **Silver** = limpio y tipado (una vista por tabla) · **Estrella** = dim_lugar, dim_tiempo, hechos_mes · **Gold** = salidas de los modelos (su método está en la nota de decisión de su fase y en `docs/metodologia/ECUACIONES.md`). Sufijos: `_est` estimado, `_flag` bandera.

| Tabla | Renglones | Qué es |
|---|---:|---|
| [`silver_afac`](#silver_afac) | 19,277 | Pasajeros por aerolínea en México, por mes (AFAC vía DataTur, 2016 → jul-2026). Nacional. |
| [`silver_clima_diario`](#silver_clima_diario) | 224,232 | Clima diario de 8 puntos de Quintana Roo (Open-Meteo, 1950 → 2026). |
| [`silver_clima_horario`](#silver_clima_horario) | 542,784 | Clima por hora de 8 punt

## 8. Lo que no existe (huecos declarados)
Un cero imposible no es un cero: es un dato que la fuente dejó de publicar. Se marca como hueco y nunca se rellena.

In [30]:
s = pd.read_parquet(RAIZ / "datos" / "silver" / "siturq")
(s.groupby("indicador").agg(renglones=("valor", "size"), huecos=("hueco_flag", "sum"),
                             ultimo_con_dato=("periodo", lambda p: p[~s.loc[p.index, "hueco_flag"]].max()))
 .sort_values("huecos", ascending=False))

,renglones,huecos,ultimo_con_dato
indicador,,,
ocupacion_hotelera,1953,693,2024-12-01
aereos_llegadas,450,114,2024-12-01
afluencia_turistas,600,15,2024-03-01
derrama_turistas,414,9,2024-03-01
derrama_visitantes,414,9,2024-03-01
centros_hospedaje,825,0,2026-07-01
cruceristas,273,0,2026-07-01
frontera_belice,180,0,2026-06-01
habitaciones,825,0,2026-07-01


In [31]:
spark.stop()